# Analisis de Pernoctaciones

- estratègia de negoci per alinear-nos amb les tendències nacionals i maximitzar les oportunitats de mercat.

## Introducción

### Comentarios

Jerarquía de Desglose (desde lo general a lo específico):

- Nacional (España)
- Comunidad/Ciudad Autónoma (Región)
- Provincia (División administrativa intermedia)
- Punto Turístico (Municipio clave seleccionado) o Zona Turística (Agrupación de municipios con identidad turística común, como "Costa del Sol" o "Pirineo Aragonés"). 

### Definiciones
Medidas:
- “Viajero” se refiere al volumen de mercado, 
- “Pernoctaciones” e refiere a la intensidad de la demanda o al uso de los alojamientos. 

KPI:
- ALOS - es la estancia media, un indicador útil para la estrategia comercial y la planificación de la capacidad (ALOS - Average Length of Overnight Stay)
        
        ALOS = Pernoctaciones / Viajero

___
- Índice de estacionalidad (Seasonality Index, SI) - valor mensual dividido por el valor mensual medio del mismo año o, preferiblemente, un índice mensual plurianual si se dispone de datos históricos suficientes.
    * valores superiores a 100 indican meses por encima de la media, 
    * los valores inferiores a 100 indican meses con baja estacionalidad.

        SI(%)= Valor mensual / Valor mensual medio del mismo año X 100

___
_YoY growth_
- YoY viajeros %
- YoY pernoctaciones %
- YoY ALOS %

Lógica de casos extremos para las tendencias:
- Mejor aceleración: mayor variación interanual YoY positiva con una base suficiente.
- Peor deterioro: mayor variación interanual YoY negativa.
- Mayor concentración estacional: relación entre el máximo y el mínimo a lo largo de los meses.
- Mayor inestabilidad: coeficiente de variación de la demanda mensual.
- Caso extremo de ALOS: aumento del número de viajeros con un ALOS estable o en descenso.

___
**Notas**
- Agregar las series mensuales por comunidad y variable.
- Comprobar la duración y la exhaustividad del historial.
- Descomponer la tendencia y la estacionalidad.
- Desestacionalizar cuando sea necesario.
- Ajustar modelos candidatos por serie; por ejemplo, un modelo de referencia estacional «naïve», ETS, SARIMA y Prophet, o regresores con refuerzo de gradiente con características de calendario.
- Realizar pruebas retrospectivas en ventanas móviles.
- Seleccionar el modelo ganador por comunidad y indicador utilizando el MAPE o el sMAPE, además del control del sesgo.
- Volver a aplicar la estacionalidad para generar la previsión final de 12 meses.
- Prever «Viajeros» y «Pernoctaciones» por separado.
- Calcular ALOS_previsión = Pernoctaciones_previsión / Viajeros_previsión.
- Señalará los resultados inverosímiles, por ejemplo, un fuerte aumento de viajeros con un ALOS en caída libre, a menos que esté justificado históricamente.
- Añada variables de calendario: mes, fecha de Semana Santa si está disponible externamente, puentes, variable ficticia de verano y variable ficticia de fin de año.
- Añada posteriormente la conciliación jerárquica para que las previsiones provinciales sumen los totales de las comunidades.

___
**Bottlenecks y oportunidades**

*Matriz de clasificación*
_ENG_
|State|Pattern|Business meaning|
|-----|-------|----------------|
|Growth opportunity|Travellers up, Pernoctaciones up faster, ALOS stable/up|Strong demand and monetizable stay depth|
|Acquisition problem|Pernoctaciones stable/up, Travellers flat/down, ALOS high|Strong stay depth but weak visitor inflow|
|Retention/stay problem|Travellers up, Pernoctaciones flat, ALOS down|Visitors arrive but stay fewer nights|
|Structural stress|Travellers down, Pernoctaciones down, ALOS down|Broad weakness, likely competitiveness or seasonality issue|

_ESP_
|Estado|Patrón|Significado empresarial|
|-----|-------|----------------|
|Oportunidad de crecimiento|Aumento de viajeros, aumento más rápido de las pernoctaciones, ALOS estable o al alza|Fuerte demanda y duración de estancia rentabilizable|
|Problema de captación|Pernoctaciones estables o al alza, número de viajeros estable o a la baja, ALOS elevado|Gran duración de las estancias, pero escasa afluencia de visitantes|
|Problema de retención/estancia|Aumento del número de viajeros, pernoctaciones estables, ALOS a la baja|Los visitantes llegan, pero se quedan menos noches|
|Tensión estructural|Descenso del número de viajeros, descenso de las pernoctaciones, descenso del ALOS|Debilidad generalizada, probablemente un problema de competitividad o estacionalidad|

**Señales bottleneck**
- Elevada concentración en temporada alta y baja base fuera de temporada.
- Fuerte crecimiento del número de viajeros, pero baja conversión en ALOS.
- La comunidad en general va bien, pero las provincias clave registran un rendimiento inferior al esperado.
- Segmentos de estancia con un volumen elevado, pero con una duración de estancia en descenso.
- Tipos de alojamiento con alta demanda, pero con un crecimiento inestable.

**Señales de oportunidad:**
- Comunidades con un número modesto de viajeros, pero con un ALOS elevado.
- Provincias con una mejora interanual y una baja estacionalidad.
- Segmentos de residentes extranjeros o nacionales con una duración media de estancia superior.
- Tipos de alojamiento con pernoctaciones resistentes fuera de temporada.

#### Forecasting
**flujo de trabajo por zona geográfica de destino:**

- Crear series mensuales de «viajeros» y «pernoctaciones».
- Comprobar que los datos estén completos, que haya un historial mínimo y que la densidad sea distinta de cero.
- Ajustar los modelos candidatos:
    * Modelo estacional «naïve».
    * ETS / Holt-Winters.
    * SARIMA.
- Realizar pruebas retrospectivas con origen móvil, por ejemplo, los últimos 12 meses.
- Comparar métricas: MAE, RMSE, sMAPE, sesgo.
- Seleccionar el modelo ganador por métrica y segmento.
- Realizar una previsión para los próximos 12 meses.
- Derivar alos_forecast a partir de los numeradores y denominadores previstos.

## Hieramientas

### Librerias

In [1]:
import sys
import warnings
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import scikit_posthocs as sp
import seaborn as sns
import statsmodels.api as sm


from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX


# Set style
plt.style.use('default')
sns.set_palette("husl")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

warnings.filterwarnings("ignore", category=FutureWarning)
sys_path = sys.executable

pio.renderers.default = "vscode"

### Functions

In [2]:
def get_notebook_dir():
    """
    Devuelve el directorio que contiene el cuaderno de Jupyter o el script de Python actual.

    En las sesiones de Jupyter de VS Code, utiliza la ruta del cuaderno; en caso contrario,
    recurre al directorio del script o al directorio de trabajo actual.

    Devuelve
    -------
    Path
        Ruta del directorio como un objeto ``pathlib.Path``.
    """
    try:
        # Variable específica disponible en el entorno Jupyter de VS Code
        return Path(__vsc_ipynb_file__).parent
    except NameError:
        # Solución alternativa para scripts .py estándar u otros entornos
        return Path(__file__).parent if "__file__" in globals() else Path.cwd()

### Carga de datos

In [3]:
notebook_dir = get_notebook_dir()

# 2. Accede a la carpeta 'Data'
# Estructura: ../Data/EOH_2021_2025/Datos_limpios
directory = notebook_dir.parent / 'Data/EOH_2021_2025/Datos_limpios'

# Definir patrones de nombres
prefix = "EOH_2021_2025"
suffix = "_limpio"
extension = ".csv"

# Buscar todos los archivos que cumplan los criterios
matching_files = [
    file for file in directory.iterdir()
    if file.is_file() and file.suffix == extension and
       file.stem.startswith(prefix) and suffix in file.stem
]

# Ordenar los archivos para garantizar un orden coherente (opcional, pero recomendable)
matching_files.sort()

# Comprobar si tenemos el número esperado de archivos (por ejemplo, 4)
if len(matching_files) >= 4:
    # Selecciona los 4 primeros archivos (o elimina [:4] para cargar todos los resultados)
    files_to_load = matching_files[:4]
    # Cargar y combinar en un único DataFrame
    df_list = [pd.read_csv(file) for file in files_to_load]
    
    print(f"Successfully loaded {len(files_to_load)} files.")
    print(f"Total rows: {len(df_list)}")
else:
    print(f"Warning: Only found {len(matching_files)} matching files (expected 4).")
    if matching_files:
        # Cargar lo que haya disponible
        df_list = [pd.read_csv(file) for file in matching_files]

    else:
        print("No matching files found.")

Successfully loaded 4 files.
Total rows: 4


In [4]:
df_alojamientos = df_list[0]
print(df_alojamientos.head(2))
print(df_alojamientos.info())

              Tipo de alojamiento  Total Nacional Comunidades y Ciudades Autónomas Residencia: Nivel 2 Viajeros y pernoctaciones  Periodo      Total   año  mes  cod_comunidad nombre_comunidad
0  Encuesta de Ocupación Hotelera  Total Nacional                              NaN                 NaN                   Viajero  2025M12  6937385.0  2025   12            NaN              NaN
1  Encuesta de Ocupación Hotelera  Total Nacional                              NaN                 NaN                   Viajero  2025M11  7413495.0  2025   11            NaN              NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36000 entries, 0 to 35999
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Tipo de alojamiento               36000 non-null  object 
 1   Total Nacional                    36000 non-null  object 
 2   Comunidades y Ciudades Autónomas  34200 non-null  object 


In [5]:
df_comunidades = df_list[2]
print(df_comunidades.head(2))
print(df_comunidades.info())

  Totales Territoriales Comunidades y Ciudades Autónomas Provincias Viajeros y pernoctaciones Residencia: Nivel 2  Periodo      Total  cod_comunidad nombre_comunidad  cod_provincia nombre_provincia  \
0        Total Nacional                              NaN        NaN                   Viajero                 NaN  2025M12  6937385.0            NaN              NaN            NaN              NaN   
1        Total Nacional                              NaN        NaN                   Viajero                 NaN  2025M11  7413495.0            NaN              NaN            NaN              NaN   

    año  mes  
0  2025   12  
1  2025   11  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25200 entries, 0 to 25199
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Totales Territoriales             25200 non-null  object 
 1   Comunidades y Ciudades Autónomas  24840 non-null

In [6]:
df_comunidades = df_comunidades.rename(columns={'Residencia: Nivel 2': 'Residencia'})
df_comunidades.loc[df_comunidades['Residencia'].isna(), 'Residencia'] = 'Total'

In [7]:
df_alojamientos = df_alojamientos.rename(columns={'Residencia: Nivel 2': 'Residencia'})
df_alojamientos.loc[df_alojamientos['Residencia'].isna(), 'Residencia'] = 'Total'

____

In [8]:
# ============================================================
# STEP 1: PREPARACIÓN DE DATOS
# ============================================================

print("=" * 80)
print("STEP 1: PREPARACIÓN Y LIMPIEZA DE DATOS")
print("=" * 80)

# Select provinces
provinces = ['Balears, Illes', 'Barcelona', 'Girona', 'Madrid',
             'Málaga', 'Sevilla', 'Valencia/València']

# ---- Procesar el conjunto de datos Viajeros ----
df_viaj_raw = df_comunidades.loc[
    df_comunidades['nombre_provincia'].isin(provinces),
].copy()

# Filtrar 'Total' in Residencia (mantener solo Residents)
df_viaj_raw = df_viaj_raw.loc[
    df_viaj_raw['Residencia'].isin(['Residentes en España', 'Residentes en el Extranjero'])
].copy()

# Deja solo el tipo Viajero
df_viaj = df_viaj_raw.loc[
    df_viaj_raw['Viajeros y pernoctaciones'] == 'Viajero'
].copy()

print(f"\nViajeros dataset: {len(df_viaj)} rows after filtering")
print(df_viaj[['Viajeros y pernoctaciones', 'Residencia', 'nombre_provincia']].value_counts())

# ---- Procesar el conjunto de datos Pernoctaciones ----
df_perno_raw = df_comunidades.loc[
    df_comunidades['nombre_provincia'].isin(provinces),
].copy()

# Filtrar 'Total' in Residencia
df_perno_raw = df_perno_raw.loc[
    df_perno_raw['Residencia'].isin(['Residentes en España', 'Residentes en el Extranjero'])
].copy()

# Deja solo el tipo Pernoctaciones
df_perno = df_perno_raw.loc[
    df_perno_raw['Viajeros y pernoctaciones'] == 'Pernoctaciones'
].copy()

print(f"\nPernoctaciones dataset: {len(df_perno)} rows after filtering")
print(df_perno[['Viajeros y pernoctaciones', 'Residencia', 'nombre_provincia']].value_counts())

# ---- Verificar la estructura de Barcelona ----
print("\n" + "=" * 80)
print("VERIFICATION: Barcelona Structure")
print("=" * 80)

for col in ['Viajeros y pernoctaciones', 'Residencia', 'nombre_provincia', 'año', 'mes']:
    print(f"\n{col}:")
    print(df_viaj[df_viaj["nombre_provincia"] == 'Barcelona'][col].value_counts())

STEP 1: PREPARACIÓN Y LIMPIEZA DE DATOS

Viajeros dataset: 840 rows after filtering
Viajeros y pernoctaciones  Residencia                   nombre_provincia 
Viajero                    Residentes en España         Balears, Illes       60
                                                        Barcelona            60
                                                        Girona               60
                                                        Madrid               60
                                                        Málaga               60
                                                        Sevilla              60
                                                        Valencia/València    60
                           Residentes en el Extranjero  Balears, Illes       60
                                                        Barcelona            60
                                                        Girona               60
                                          

In [9]:
# ============================================================
# STEP 2: CREAR UN CONJUNTO DE DATOS COMBINADO CON ETIQUETA DE TIPO
# ============================================================

print("\n" + "=" * 80)
print("STEP 2: CREAR UN CONJUNTO DE DATOS COMBINADO CON ETIQUETA DE TIPO")
print("=" * 80)

# Añadir indicador de tipo
df_viaj['type'] = 'Viajero'
df_perno['type'] = 'Pernoctaciones'

# Combinar conjuntos de datos
df_all = pd.concat([df_viaj, df_perno], ignore_index=True)

# Asegúrese de que Total sea un valor numérico
df_all['Total'] = pd.to_numeric(df_all['Total'], errors='coerce')
df_all = df_all.dropna(subset=['Total', 'nombre_provincia', 'type'])

# Convertir el año y el mes a valores numéricos para su análisis
df_all['año'] = pd.to_numeric(df_all['año'], errors='coerce')
df_all['mes'] = pd.to_numeric(df_all['mes'], errors='coerce')

print(f"\nCombined dataset: {len(df_all)} rows")
print(f"Types: {df_all['type'].value_counts().to_dict()}")
print(f"Provinces: {df_all['nombre_provincia'].unique()}")
print(f"Residencia: {df_all['Residencia'].unique()}")
print(f"Years: {df_all['año'].unique()}")
print(f"Months: {df_all['mes'].unique()}")

# Guardar para pasos posteriores
df_all.to_csv('output/combined_analysis_data.csv', index=False)
print("\n✓ Data saved to output/combined_analysis_data.csv")


STEP 2: CREAR UN CONJUNTO DE DATOS COMBINADO CON ETIQUETA DE TIPO

Combined dataset: 1680 rows
Types: {'Viajero': 840, 'Pernoctaciones': 840}
Provinces: ['Málaga' 'Sevilla' 'Balears, Illes' 'Barcelona' 'Girona'
 'Valencia/València' 'Madrid']
Residencia: ['Residentes en España' 'Residentes en el Extranjero']
Years: [2025 2024 2023 2022 2021]
Months: [12 11 10  9  8  7  6  5  4  3  2  1]

✓ Data saved to output/combined_analysis_data.csv


In [10]:
# ============================================================
# STEP 3: ESTADÍSTICAS DESCRIPTIVAS POR PARÁMETRO
# ============================================================

print("\n" + "=" * 80)
print("STEP 3: ESTADÍSTICAS DESCRIPTIVAS POR PARÁMETRO")
print("=" * 80)

# Parameters to analyze
parameters = ['nombre_provincia', 'Residencia', 'año', 'mes']

descriptive_results = {}

for param in parameters:
    print(f"\n" + "-" * 80)
    print(f"PARAMETER: {param}")
    print("-" * 80)
    
    # Group by parameter AND type
    summary = (
        df_all.groupby([param, 'type'])['Total']
        .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        .reset_index()
        .sort_values([param, 'type'])
    )
    
    descriptive_results[param] = summary
    
    # Display as table
    print(summary.to_string(index=False))
    
    # Save
    summary.to_csv(f'output/descriptive_by_{param}.csv', index=False)

# Save all results
pd.to_pickle(descriptive_results, 'output/descriptive_results.pkl')
print("\n✓ All descriptive statistics saved")


STEP 3: ESTADÍSTICAS DESCRIPTIVAS POR PARÁMETRO

--------------------------------------------------------------------------------
PARAMETER: nombre_provincia
--------------------------------------------------------------------------------
 nombre_provincia           type  count         mean   median          std      min        max
   Balears, Illes Pernoctaciones    120 2.223998e+06 544427.5 3.273257e+06  10944.0 10532565.0
   Balears, Illes        Viajero    120 4.355717e+05 167472.0 5.838856e+05   3074.0  1855591.0
        Barcelona Pernoctaciones    120 1.218237e+06 645357.5 9.630939e+05  77219.0  3471711.0
        Barcelona        Viajero    120 4.695037e+05 318684.5 2.974167e+05  32395.0  1131857.0
           Girona Pernoctaciones    120 4.658937e+05 277080.0 4.345388e+05  14160.0  1685607.0
           Girona        Viajero    120 1.626790e+05 147164.5 1.043357e+05   7823.0   406676.0
           Madrid Pernoctaciones    120 9.537891e+05 976748.5 3.121212e+05 109984.0  1621038.0


In [11]:
# ============================================================
# STEP 4: PRUEBAS DE ASOCIACIÓN ESTADÍSTICA
# ============================================================

print("\n" + "=" * 80)
print("STEP 4: PRUEBAS DE KRUSKAL-WALLIS DE ASOCIACIÓN")
print("=" * 80)

kw_results = {}

for param in parameters:
    print(f"\n" + "-" * 80)
    print(f"Testing association: Total vs {param}")
    print("-" * 80)
    
    # Group by parameter
    groups = [g['Total'].values for _, g in df_all.groupby(param)]
    
    # Run Kruskal-Wallis
    kw_stat, kw_p = stats.kruskal(*groups)
    
    # Check assumptions
    shapiro_p_values = []
    for group_name, group_data in df_all.groupby(param):
        x = group_data['Total'].dropna().values
        if len(x) >= 3 and len(x) <= 5000:
            _, p = stats.shapiro(x)
            shapiro_p_values.append(p)
    
    shapiro_min_p = min(shapiro_p_values) if shapiro_p_values else None
    
    # Levene test for equal variance
    levene_stat, levene_p = stats.levene(*groups, center='median')
    
    # Store results
    kw_results[param] = {
        'statistic': kw_stat,
        'p_value': kw_p,
        'shapiro_min_p': shapiro_min_p,
        'levene_stat': levene_stat,
        'levene_p': levene_p,
        'n_groups': len(groups)
    }
    
    # Print interpretation
    print(f"\nKruskal-Wallis H-statistic: {kw_stat:.4f}")
    print(f"p-value: {kw_p:.10e}")
    print(f"Min Shapiro-Wilk p: {shapiro_min_p:.6f}")
    print(f"Levene test (equal variance): p = {levene_p:.6f}")
    
    # Interpretation
    if kw_p < 0.05:
        print(f"✓ SIGNIFICATIVO: Total is associated with {param} (p < 0.05)")
    else:
        print(f"✗ NO SIGNIFICATIVO: No association between Total and {param} (p >= 0.05)")
    
    print()

# Save results
with open('output/kw_results.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items()} for k, v in kw_results.items()}, f, indent=2)

print("\n✓ Kruskal-Wallis results saved")


STEP 4: PRUEBAS DE KRUSKAL-WALLIS DE ASOCIACIÓN

--------------------------------------------------------------------------------
Testing association: Total vs nombre_provincia
--------------------------------------------------------------------------------

Kruskal-Wallis H-statistic: 431.8090
p-value: 4.0307419771e-90
Min Shapiro-Wilk p: 0.000000
Levene test (equal variance): p = 0.000000
✓ SIGNIFICATIVO: Total is associated with nombre_provincia (p < 0.05)


--------------------------------------------------------------------------------
Testing association: Total vs Residencia
--------------------------------------------------------------------------------

Kruskal-Wallis H-statistic: 44.3395
p-value: 2.7609130673e-11
Min Shapiro-Wilk p: 0.000000
Levene test (equal variance): p = 0.000000
✓ SIGNIFICATIVO: Total is associated with Residencia (p < 0.05)


--------------------------------------------------------------------------------
Testing association: Total vs año
--------------

In [12]:
# ============================================================
# STEP 5: PRUEBAS POST HOC PARA PARÁMETROS SIGNIFICATIVOS
# ============================================================

print("\n" + "=" * 80)
print("STEP 5: PRUEBAS POST-HOC (Dunn's Test)")
print("=" * 80)

# First debug: Check the structure
print("\n=== DEBUG: Checking Dunn's test output ===")
dunn_debug = sp.posthoc_dunn(
    df_all,
    val_col='Total',
    group_col='nombre_provincia',
    p_adjust='bonferroni'
)
print("Shape:", dunn_debug.shape)
print("Columns:", dunn_debug.columns)
print("Index:", dunn_debug.index)
print("\nDunn's test output:")
print(dunn_debug)
print("=== END DEBUG ===\n")

posthoc_results = {}

for param in parameters:
    if kw_results[param]['p_value'] >= 0.05:
        print(f"\nSkipping {param}: Kruskal-Wallis not significant")
        continue

    print(f"\n" + "-" * 80)
    print(f"Post-hoc for: {param}")
    print("-" * 80)

    # Dunn's test
    dunn = sp.posthoc_dunn(
        df_all, 
        val_col='Total',
        group_col=param,
        p_adjust='bonferroni'
    )
    
    posthoc_results[param] = dunn
    
    # Print full table
    print("\nFull Dunn's test results:")
    print(dunn.to_string())
    
    # Use iterrows() to iterate safely
    significant_pairs = []
    for idx, row in dunn.iterrows():
        # row is a Series, access by column position
        # Typically: column 0 = group1, column 1 = group2, column 2 = p_val, column 3 = p_adj
        # Or check if there are column names
        if len(dunn.columns) >= 4:
            group1 = row.iloc[0]
            group2 = row.iloc[1]
            p_adj = row.iloc[3]  # Adjusted p-value is typically 4th column
        elif len(dunn.columns) == 2:
            # Fall back: assume first column is group info, second is p_adj
            group1, group2 = idx if isinstance(idx, tuple) else (idx, None)
            p_adj = row.iloc[1]
        else:
            # Try to get from row values
            row_list = row.tolist()
            group1 = row_list[0] if len(row_list) > 0 else None
            group2 = row_list[1] if len(row_list) > 1 else None
            p_adj = row_list[-1]  # Last value is usually p_adj
        
        if p_adj < 0.05:
            significant_pairs.append({
                'group1': group1,
                'group2': group2,
                'p_adj': p_adj
            })
    
    print(f"\nSignificant pairs (p < 0.05): {len(significant_pairs)}")
    for pair in significant_pairs[:10]:
        print(f"  {pair['group1']} vs {pair['group2']}: p = {pair['p_adj']:.6f}")
    
    if len(significant_pairs) > 10:
        print(f"  ... and {len(significant_pairs) - 10} more")
    
    print()

# Save
pd.to_pickle(posthoc_results, 'output/posthoc_results.pkl')
print("\n✓ Post-hoc results saved")


STEP 5: PRUEBAS POST-HOC (Dunn's Test)

=== DEBUG: Checking Dunn's test output ===
Shape: (7, 7)
Columns: Index(['Balears, Illes', 'Barcelona', 'Girona', 'Madrid', 'Málaga', 'Sevilla', 'Valencia/València'], dtype='object')
Index: Index(['Balears, Illes', 'Barcelona', 'Girona', 'Madrid', 'Málaga', 'Sevilla', 'Valencia/València'], dtype='object')

Dunn's test output:
                   Balears, Illes     Barcelona        Girona        Madrid        Málaga       Sevilla  Valencia/València
Balears, Illes       1.000000e+00  1.020234e-11  2.680550e-04  3.251730e-18  1.000000e+00  4.932677e-08       9.056655e-06
Barcelona            1.020234e-11  1.000000e+00  9.357124e-30  1.000000e+00  3.479067e-07  1.822010e-38       2.323692e-33
Girona               2.680550e-04  9.357124e-30  1.000000e+00  1.180832e-39  5.671816e-08  1.000000e+00       1.000000e+00
Madrid               3.251730e-18  1.000000e+00  1.180832e-39  1.000000e+00  1.864907e-12  1.269057e-49       8.452959e-44
Málaga          

In [13]:
# ============================================================
# STEP 6: COMPARACIÓN ENTRE LA MEDIA Y LA MEDIANA POR PARÁMETRO
# ============================================================

print("\n" + "=" * 80)
print("STEP 6: COMPARACIÓN ENTRE LA MEDIA Y LA MEDIANA")
print("=" * 80)

mean_median_comparison = {}

for param in parameters:
    print(f"\n" + "-" * 80)
    print(f"{param}: Mean vs Median by Type")
    print("-" * 80)
    
    # Calculate mean/median ratio (indicator of skewness)
    analysis = (
        df_all.groupby([param, 'type'])['Total']
        .agg(['mean', 'median'])
        .reset_index()
    )
    
    analysis['mean_median_ratio'] = analysis['mean'] / analysis['median']
    analysis['skewness_indicator'] = np.where(
        analysis['mean_median_ratio'] > 1.1, 'Right-skewed',
        np.where(analysis['mean_median_ratio'] < 0.9, 'Left-skewed', 'Symmetric')
    )
    
    mean_median_comparison[param] = analysis
    
    print(analysis.to_string(index=False))
    
    # Save
    analysis.to_csv(f'output/mean_median_by_{param}.csv', index=False)

print("\n✓ Mean vs median comparison saved")


STEP 6: COMPARACIÓN ENTRE LA MEDIA Y LA MEDIANA

--------------------------------------------------------------------------------
nombre_provincia: Mean vs Median by Type
--------------------------------------------------------------------------------
 nombre_provincia           type         mean   median  mean_median_ratio skewness_indicator
   Balears, Illes Pernoctaciones 2.223998e+06 544427.5           4.085022       Right-skewed
   Balears, Illes        Viajero 4.355717e+05 167472.0           2.600863       Right-skewed
        Barcelona Pernoctaciones 1.218237e+06 645357.5           1.887693       Right-skewed
        Barcelona        Viajero 4.695037e+05 318684.5           1.473255       Right-skewed
           Girona Pernoctaciones 4.658937e+05 277080.0           1.681441       Right-skewed
           Girona        Viajero 1.626790e+05 147164.5           1.105423       Right-skewed
           Madrid Pernoctaciones 9.537891e+05 976748.5           0.976494          Symmetric
   

In [14]:
# ============================================================
# STEP 8: LOS CASOS MÁS CONTRASTANTES
# ============================================================

print("\n" + "=" * 80)
print("STEP 8: LOS CASOS MÁS CONTRASTANTES")
print("=" * 80)

contrasting_cases = {}

for param in parameters:
    print(f"\n" + "-" * 80)
    print(f"Contrasting cases for: {param}")
    print("-" * 80)
    
    # Get mean by parameter and type
    means = (
        df_all.groupby([param, 'type'])['Total']
        .mean()
        .reset_index()
    )
    
    # For each type, find highest and lowest
    for type_val in ['Viajero', 'Pernoctaciones']:
        type_data = means[means['type'] == type_val]
        
        highest = type_data.nlargest(1, 'Total')
        lowest = type_data.nsmallest(1, 'Total')
        
        contrasting_cases[param] = {
            'type': type_val,
            'highest': {
                'group': highest.iloc[0][param],
                'mean': highest.iloc[0]['Total']
            },
            'lowest': {
                'group': lowest.iloc[0][param],
                'mean': lowest.iloc[0]['Total']
            },
            'ratio': highest.iloc[0]['Total'] / lowest.iloc[0]['Total']
        }
        
        print(f"\n{type_val}:")
        print(f"  Highest: {contrasting_cases[param]['highest']['group']} = {contrasting_cases[param]['highest']['mean']:.2f}")
        print(f"  Lowest: {contrasting_cases[param]['lowest']['group']} = {contrasting_cases[param]['lowest']['mean']:.2f}")
        print(f"  Ratio (highest/lowest): {contrasting_cases[param]['ratio']:.2f}")

# Save
pd.to_pickle(contrasting_cases, 'output/contrasting_cases.pkl')
print("\n✓ Contrasting cases saved")


STEP 8: LOS CASOS MÁS CONTRASTANTES

--------------------------------------------------------------------------------
Contrasting cases for: nombre_provincia
--------------------------------------------------------------------------------

Viajero:
  Highest: Madrid = 472366.58
  Lowest: Valencia/València = 137781.62
  Ratio (highest/lowest): 3.43

Pernoctaciones:
  Highest: Balears, Illes = 2223998.25
  Lowest: Sevilla = 277752.33
  Ratio (highest/lowest): 8.01

--------------------------------------------------------------------------------
Contrasting cases for: Residencia
--------------------------------------------------------------------------------

Viajero:
  Highest: Residentes en el Extranjero = 366300.14
  Lowest: Residentes en España = 219212.76
  Ratio (highest/lowest): 1.67

Pernoctaciones:
  Highest: Residentes en el Extranjero = 1328238.49
  Lowest: Residentes en España = 460774.22
  Ratio (highest/lowest): 2.88

--------------------------------------------------------

In [15]:
# ============================================================
# STEP 9: ANÁLISIS DEL VALOR EMPRESARIAL
# ============================================================

print("\n" + "=" * 80)
print("STEP 9: VALOR EMPRESARIAL: DEMANDA Y OFERTA")
print("=" * 80)

# Calculate demand metrics
demand_analysis = (
    df_all.groupby(['nombre_provincia', 'type'])['Total']
    .agg(['sum', 'mean', 'median'])
    .reset_index()
)

# Calculate offer metrics (assuming offer = capacity to accommodate)
# For this analysis, we'll use mean as "typical offer" and total as "total demand"
demand_analysis['demand_total'] = demand_analysis['sum']
demand_analysis['offer_typical'] = demand_analysis['mean']
demand_analysis['demand_offer_ratio'] = demand_analysis['demand_total'] / demand_analysis['offer_typical']

print("\nDemand and Offer by Province:")
print(demand_analysis.to_string(index=False))

# Identify high-demand, high-value provinces
high_demand = demand_analysis.nlargest(3, 'demand_total')
high_value = demand_analysis.nlargest(3, 'offer_typical')

print("\n" + "-" * 80)
print("TOP 3: Highest Total Demand")
print("-" * 80)
for _, row in high_demand.iterrows():
    print(f"  {row['nombre_provincia']} ({row['type']}): {row['demand_total']:.2f}")

print("\n" + "-" * 80)
print("TOP 3: Highest Typical Offer (Mean)")
print("-" * 80)
for _, row in high_value.iterrows():
    print(f"  {row['nombre_provincia']} ({row['type']}): {row['offer_typical']:.2f}")

# Business interpretation
print("\n" + "-" * 80)
print("BUSINESS INTERPRETATION")
print("-" * 80)

for province in demand_analysis['nombre_provincia'].unique():
    prov_data = demand_analysis[demand_analysis['nombre_provincia'] == province]
    
    viaj_data = prov_data[prov_data['type'] == 'Viajero']
    perno_data = prov_data[prov_data['type'] == 'Pernoctaciones']
    
    if viaj_data.empty or perno_data.empty:
        continue
    
    viaj_demand = viaj_data['demand_total'].iloc[0]
    perno_demand = perno_data['demand_total'].iloc[0]
    
    conversion_rate = perno_demand / viaj_demand if viaj_demand > 0 else 0
    
    print(f"\n{province}:")
    print(f"  Viajeros demand: {viaj_demand:.2f}")
    print(f"  Pernoctaciones demand: {perno_demand:.2f}")
    print(f"  Conversion rate (Pernoctaciones/Viajeros): {conversion_rate:.2f}")
    
    if conversion_rate > 1.5:
        print(f"  → HIGH conversion: Strong lodging demand")
    elif conversion_rate < 0.5:
        print(f"  → LOW conversion: Weak lodging demand relative to travelers")
    else:
        print(f"  → MODERATE conversion: Balanced demand")

# Save
demand_analysis.to_csv('output/business_value_analysis.csv', index=False)
print("\n✓ Business value analysis saved")


STEP 9: VALOR EMPRESARIAL: DEMANDA Y OFERTA

Demand and Offer by Province:
 nombre_provincia           type         sum         mean   median  demand_total  offer_typical  demand_offer_ratio
   Balears, Illes Pernoctaciones 266879790.0 2.223998e+06 544427.5   266879790.0   2.223998e+06               120.0
   Balears, Illes        Viajero  52268604.0 4.355717e+05 167472.0    52268604.0   4.355717e+05               120.0
        Barcelona Pernoctaciones 146188457.0 1.218237e+06 645357.5   146188457.0   1.218237e+06               120.0
        Barcelona        Viajero  56340439.0 4.695037e+05 318684.5    56340439.0   4.695037e+05               120.0
           Girona Pernoctaciones  55907244.0 4.658937e+05 277080.0    55907244.0   4.658937e+05               120.0
           Girona        Viajero  19521474.0 1.626790e+05 147164.5    19521474.0   1.626790e+05               120.0
           Madrid Pernoctaciones 114454686.0 9.537891e+05 976748.5   114454686.0   9.537891e+05               12

In [16]:
# ============================================================
# STEP 10: INFORME RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("STEP 10: INFORME RESUMEN FINAL")
print("=" * 80)

report = []

report.append("=" * 80)
report.append("ANÁLISIS DE ASOCIACIONES: Pernoctaciones & Viajeros")
report.append("=" * 80)

report.append("\n" + "-" * 80)
report.append("SIGNIFICACIÓN ESTADÍSTICA (Kruskal-Wallis)")
report.append("-" * 80)

for param in parameters:
    kw = kw_results[param]
    sig = "✓ SIGNIFICANT" if kw['p_value'] < 0.05 else "✗ NOT SIGNIFICANT"
    report.append(f"\n{param}:")
    report.append(f"  H-statistic: {kw['statistic']:.4f}")
    report.append(f"  p-value: {kw['p_value']:.10e}")
    report.append(f"  Result: {sig}")

report.append("\n" + "-" * 80)
report.append("MOST CONTRASTING CASES")
report.append("-" * 80)

for param, cases in contrasting_cases.items():
    report.append(f"\n{param}:")
    report.append(f"  {cases['type']} Highest: {cases['highest']['group']} = {cases['highest']['mean']:.2f}")
    report.append(f"  {cases['type']} Lowest: {cases['lowest']['group']} = {cases['lowest']['mean']:.2f}")
    report.append(f"  Ratio: {cases['ratio']:.2f}")

report.append("\n" + "-" * 80)
report.append("BUSINESS VALUE SUMMARY")
report.append("-" * 80)

top_provinces = demand_analysis.nlargest(3, 'demand_total')
for _, row in top_provinces.iterrows():
    report.append(f"\n{row['nombre_provincia']} ({row['type']}):")
    report.append(f"  Total Demand: {row['demand_total']:.2f}")
    report.append(f"  Typical Offer: {row['offer_typical']:.2f}")

report.append("\n" + "=" * 80)
report.append("END OF REPORT")
report.append("=" * 80)

# Print report
for line in report:
    print(line)

# Save report
with open('output/final_summary_report.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(report))

print("\n✓ Final report saved to output/final_summary_report.txt")


STEP 10: INFORME RESUMEN FINAL
ANÁLISIS DE ASOCIACIONES: Pernoctaciones & Viajeros

--------------------------------------------------------------------------------
SIGNIFICACIÓN ESTADÍSTICA (Kruskal-Wallis)
--------------------------------------------------------------------------------

nombre_provincia:
  H-statistic: 431.8090
  p-value: 4.0307419771e-90
  Result: ✓ SIGNIFICANT

Residencia:
  H-statistic: 44.3395
  p-value: 2.7609130673e-11
  Result: ✓ SIGNIFICANT

año:
  H-statistic: 147.3225
  p-value: 7.6282959668e-31
  Result: ✓ SIGNIFICANT

mes:
  H-statistic: 276.1800
  p-value: 9.0038570674e-53
  Result: ✓ SIGNIFICANT

--------------------------------------------------------------------------------
MOST CONTRASTING CASES
--------------------------------------------------------------------------------

nombre_provincia:
  Pernoctaciones Highest: Balears, Illes = 2223998.25
  Pernoctaciones Lowest: Sevilla = 277752.33
  Ratio: 8.01

Residencia:
  Pernoctaciones Highest: Reside

It is a noticeable difference between mean and median values, especially in case of 'Balears, Illes' and 'Girona'. This gives the idea of strong skewness of the data

In [17]:
# -----------------------------
# 4) Kruskal-Wallis test (main test)
# -----------------------------
groups = [g['Total'].values for _, g in df_all.groupby('nombre_provincia')]
kw_stat, kw_p = stats.kruskal(*groups)

print("\nKruskal-Wallis test (non-parametric):")
print(f"statistic = {kw_stat:.4f}, p-value = {kw_p:.10e}")


Kruskal-Wallis test (non-parametric):
statistic = 431.8090, p-value = 4.0307419771e-90


Aplicación de la prueba de Kruskal-Wallis como alternativa no paramétrica al ANOVA unidireccional. Permite comprobar si tres o más grupos independientes tienen la misma mediana poblacional (distribución). Se utiliza cuando no se cumplen los supuestos de normalidad o de varianza igual.

In [18]:
# -----------------------------
# 5) Welch's ANOVA (robust ANOVA)
# -----------------------------
# statsmodels does not have Welch's ANOVA directly, use pingouin if available,
# or implement via scipy's f_oneway with custom variance adjustment.
try:
    import pingouin as pg
    welch = pg.welch_anova(data=df_all, dv='Total', between='nombre_provincia')
    print("\nWelch's ANOVA:")
    print(welch)
except ImportError:
    print("\npingouin not installed; skipping Welch's ANOVA.")
    print("Install with: pip install pingouin")



Welch's ANOVA:
             Source  ddof1       ddof2           F          p_unc       np2
0  nombre_provincia      6  709.026941  119.772559  2.813119e-104  0.116687


In [19]:
# -----------------------------
# 6) Classical one-way ANOVA (for reference)
# -----------------------------
model = ols('Total ~ C(nombre_provincia)', data=df_all).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("\nOne-way ANOVA (classical, for reference):")
print(anova_table)

# Effect size
ss_between = anova_table.loc['C(nombre_provincia)', 'sum_sq']
ss_total = ss_between + anova_table.loc['Residual', 'sum_sq']
eta_sq = ss_between / ss_total
print(f"\nEta-squared (classical ANOVA): {eta_sq:.4f}")


One-way ANOVA (classical, for reference):
                           sum_sq      df          F        PR(>F)
C(nombre_provincia)  2.360151e+14     6.0  36.834162  4.096403e-42
Residual             1.786627e+15  1673.0        NaN           NaN

Eta-squared (classical ANOVA): 0.1167


In [20]:
# -----------------------------
# 7) Post-hoc tests
# -----------------------------
# 7a) Tukey HSD (for ANOVA-scale differences)
tukey = pairwise_tukeyhsd(
    endog=df_all['Total'],
    groups=df_all['nombre_provincia'],
    alpha=0.05
)
print("\nTukey HSD post-hoc:")
print(tukey.summary())



Tukey HSD post-hoc:
                  Multiple Comparison of Means - Tukey HSD, FWER=0.05                  
    group1           group2         meandiff   p-adj      lower        upper     reject
---------------------------------------------------------------------------------------
Balears, Illes         Barcelona   -485914.575    0.0  -764387.6193 -207441.5307   True
Balears, Illes            Girona   -1015498.65    0.0 -1293971.6943 -737025.6057   True
Balears, Illes            Madrid  -616707.1625    0.0  -895180.2068 -338234.1182   True
Balears, Illes            Málaga  -813980.8333    0.0 -1092453.8776  -535507.789   True
Balears, Illes           Sevilla -1121571.2542    0.0 -1400044.2985 -843098.2099   True
Balears, Illes Valencia/València -1099402.5375    0.0 -1377875.5818 -820929.4932   True
     Barcelona            Girona   -529584.075    0.0  -808057.1193 -251111.0307   True
     Barcelona            Madrid  -130792.5875 0.8092  -409265.6318  147680.4568  False
     Barcel

In [21]:

# 7b) Pairwise Wilcoxon with Bonferroni correction
prov_list = df_all['nombre_provincia'].unique()
n_prov = len(prov_list)
pairwise_results = []

for i in range(n_prov):
    for j in range(i+1, n_prov):
        a = df_all.loc[df_all['nombre_provincia'] == prov_list[i], 'Total']
        b = df_all.loc[df_all['nombre_provincia'] == prov_list[j], 'Total']
        stat, p = stats.mannwhitneyu(a, b, alternative='two-sided')
        pairwise_results.append({
            'group1': prov_list[i],
            'group2': prov_list[j],
            'p_raw': p
        })

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df['p_bonf'] = pairwise_df['p_raw'] * (n_prov * (n_prov - 1) / 2)
pairwise_df['p_bonf'] = pairwise_df['p_bonf'].clip(upper=1.0)

print("\nPairwise Wilcoxon (Mann-Whitney U) with Bonferroni correction:")
print(pairwise_df.sort_values('p_bonf'))




Pairwise Wilcoxon (Mann-Whitney U) with Bonferroni correction:
               group1             group2         p_raw        p_bonf
10            Sevilla             Madrid  6.591465e-61  1.384208e-59
20  Valencia/València             Madrid  1.106578e-56  2.323814e-55
7             Sevilla          Barcelona  1.064827e-46  2.236137e-45
16          Barcelona  Valencia/València  1.189444e-40  2.497832e-39
19             Girona             Madrid  5.336902e-37  1.120749e-35
15          Barcelona             Girona  4.126216e-29  8.665053e-28
0              Málaga            Sevilla  5.166685e-17  1.085004e-15
5              Málaga             Madrid  1.760047e-16  3.696099e-15
4              Málaga  Valencia/València  9.402223e-13  1.974467e-11
3              Málaga             Girona  3.712573e-10  7.796403e-09
2              Málaga          Barcelona  5.640087e-10  1.184418e-08
14     Balears, Illes             Madrid  2.906346e-08  6.103327e-07
11     Balears, Illes          Barcelon

In [22]:
df_all.head()

,Totales Territoriales,Comunidades y Ciudades Autónomas,Provincias,Viajeros y pernoctaciones,Residencia,Periodo,Total,cod_comunidad,nombre_comunidad,cod_provincia,nombre_provincia,año,mes,type
0,Total Nacional,01 Andalucía,29 Málaga,Viajero,Residentes en España,2025M12,140369.0,1.0,Andalucía,29.0,Málaga,2025,12,Viajero
1,Total Nacional,01 Andalucía,29 Málaga,Viajero,Residentes en España,2025M11,148879.0,1.0,Andalucía,29.0,Málaga,2025,11,Viajero
2,Total Nacional,01 Andalucía,29 Málaga,Viajero,Residentes en España,2025M10,163815.0,1.0,Andalucía,29.0,Málaga,2025,10,Viajero
3,Total Nacional,01 Andalucía,29 Málaga,Viajero,Residentes en España,2025M09,194673.0,1.0,Andalucía,29.0,Málaga,2025,9,Viajero
4,Total Nacional,01 Andalucía,29 Málaga,Viajero,Residentes en España,2025M08,291484.0,1.0,Andalucía,29.0,Málaga,2025,8,Viajero


# ============================================================
# TOURISM DEMAND ANALYSIS NOTEBOOK
Áreas geográficas prioritarias:
- Balears, Illes, Barcelona, Girona, Madrid, Málaga, Sevilla, Valencia/València
Período: 
- 2021-2025: historial
# ============================================================

=========================
### 1. Config
=========================

In [23]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# PROVINCIAS
TARGET_GEOS = [
    "Balears, Illes",
    "Barcelona",
    "Girona",
    "Madrid",
    "Málaga",
    "Sevilla",
    "Valencia/València"
]

# COMUNIDADES
FOCUS_COMMUNITIES = [
    "Andalucía",
    "Balears, Illes",
    "Cataluña",
    "Comunitat Valenciana",
    "Madrid, Comunidad de"
]

FORECAST_HORIZON = 12
SEASONAL_PERIOD = 12
MIN_HISTORY = 36
MAX_MISSING_PCT = 0.15
MIN_POSITIVE_OBS = 24
LATEST_YEAR = 2025

COLOR_PRIMARY = "#0B6E75"
COLOR_SECONDARY = "#2F4858"
COLOR_ACCENT = "#D97A00"
COLOR_SUCCESS = "#4C956C"
COLOR_RISK = "#B23A48"

sns.set_theme(style="whitegrid", context="talk")

=========================
### 2. Utility functions
=========================

In [24]:
def ensure_datetime_from_period(period_series: pd.Series) -> pd.Series:
    """
    Convert strings like 2025M12 to pandas Timestamp at month start.
    """
    
    s = period_series.astype(str).str.strip()
    year = s.str.extract(r"(\d{4})")[0].astype(int)
    month = s.str.extract(r"M(\d{1,2})")[0].astype(int)
    return pd.to_datetime(dict(year=year, month=month, day=1))


def normalize_text(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def standardize_measure_name(s: pd.Series) -> pd.Series:
    s = normalize_text(s)
    mapping = {
        "Viajero": "Viajero",
        "Viajeros": "Viajero",
        "Pernoctación": "Pernoctaciones",
        "Pernoctaciones": "Pernoctaciones",
    }
    return s.replace(mapping)


def standardize_residencia(s: pd.Series) -> pd.DataFrame:
    """
    Preserve detail and derive broad groups.
    Adjust mapping to your real labels once inspected.
    """
    x = normalize_text(s)
    x_clean = x.str.lower()

    domestic_keywords = [
        "españa", "espana", "residentes en españa", "residentes en espana", "nacional", "residentes"
    ]
    international_keywords = [
        "extranjero", "extranjeros", "resto de la ue", "ue", "internacional", "no residentes"
    ]

    broad = pd.Series(pd.NA, index=x.index, dtype="string")

    broad[x.isna()] = "Total"
    for kw in domestic_keywords:
        broad[x_clean.fillna("").str.contains(kw, regex=False)] = "Doméstico"
    for kw in international_keywords:
        broad[x_clean.fillna("").str.contains(kw, regex=False)] = "Internacional"

    broad = broad.fillna("Detalle")
    return pd.DataFrame({
        "residencia_detail": x,
        "residencia_group": broad
    })


def safe_divide(num, den):
    return np.where((den.notna()) & (den != 0), num / den, np.nan)


def add_time_features(df: pd.DataFrame, date_col: str = "fecha") -> pd.DataFrame:
    out = df.copy()
    out["año"] = out[date_col].dt.year
    out["mes"] = out[date_col].dt.month
    out["quarter"] = out[date_col].dt.quarter
    out["month_name"] = out[date_col].dt.month_name(locale=None)
    out["month_id"] = out["año"] * 100 + out["mes"]
    return out


def winsorize_series(s: pd.Series, lower=0.01, upper=0.99) -> pd.Series:
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def robust_zscore(s: pd.Series) -> pd.Series:
    med = np.nanmedian(s)
    mad = np.nanmedian(np.abs(s - med))
    if mad == 0 or np.isnan(mad):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return 0.6745 * (s - med) / mad


def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred))
    mask = denom != 0
    return 100 * np.mean(2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]) if mask.any() else np.nan


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def make_complete_month_index(df, date_col="fecha", group_cols=None, value_cols=None):
    if group_cols is None:
        group_cols = []
    if value_cols is None:
        value_cols = []

    pieces = []
    for keys, g in df.groupby(group_cols, dropna=False):
        g = g.sort_values(date_col).copy()
        idx = pd.date_range(g[date_col].min(), g[date_col].max(), freq="MS")
        base = pd.DataFrame({date_col: idx})
        if not isinstance(keys, tuple):
            keys = (keys,)
        for col, val in zip(group_cols, keys):
            base[col] = val
        g2 = base.merge(g, on=group_cols + [date_col], how="left")
        pieces.append(g2)

    out = pd.concat(pieces, ignore_index=True) if pieces else df.copy()
    return out


=========================
### 3. Data preparation
=========================

In [25]:
def prepare_df_alojamientos(df_alojamientos: pd.DataFrame) -> pd.DataFrame:
    df = df_alojamientos.copy()

    rename_map = {
        "Tipo de alojamiento": "tipo_alojamiento",
        "Residencia": "residencia_raw",
        "Viajeros y pernoctaciones": "measure_name",
        "Total": "total",
        "Periodo": "periodo",
        "nombre_comunidad": "nombre_comunidad"
    }
    df = df.rename(columns=rename_map)

    keep_cols = [
        "tipo_alojamiento", "residencia_raw", "measure_name", "total",
        "periodo", "año", "mes", "nombre_comunidad"
    ]
    df = df[keep_cols].copy()

    df["tipo_alojamiento"] = normalize_text(df["tipo_alojamiento"])
    df["residencia_raw"] = normalize_text(df["residencia_raw"])
    df["measure_name"] = standardize_measure_name(df["measure_name"])
    df["total"] = pd.to_numeric(df["total"], errors="coerce")
    df["fecha"] = ensure_datetime_from_period(df["periodo"])
    df["nombre_comunidad"] = normalize_text(df["nombre_comunidad"])

    residencia_df = standardize_residencia(df["residencia_raw"])
    df = pd.concat([df, residencia_df], axis=1)

    df["geo_level"] = np.where(df["nombre_comunidad"].notna(), "comunidad", "nacional")
    df["geo_name"] = df["nombre_comunidad"].fillna("Total Nacional")
    df["source_table"] = "df_alojamientos"

    df = add_time_features(df, "fecha")

    return df


def prepare_df_comunidades(df_comunidades: pd.DataFrame) -> pd.DataFrame:
    df = df_comunidades.copy()

    rename_map = {
        "nombre_comunidad": "nombre_comunidad",
        "nombre_provincia": "nombre_provincia",
        "Viajeros y pernoctaciones": "measure_name",
        "Residencia": "residencia_raw",
        "Total": "total",
        "Periodo": "periodo"
    }
    df = df.rename(columns=rename_map)

    keep_cols = [
        "nombre_comunidad", "nombre_provincia", "measure_name",
        "residencia_raw", "total", "periodo", "año", "mes"
    ]
    df = df[keep_cols].copy()

    df["nombre_comunidad"] = normalize_text(df["nombre_comunidad"])
    df["nombre_provincia"] = normalize_text(df["nombre_provincia"])
    df["measure_name"] = standardize_measure_name(df["measure_name"])
    df["residencia_raw"] = normalize_text(df["residencia_raw"])
    df["total"] = pd.to_numeric(df["total"], errors="coerce")
    df["fecha"] = ensure_datetime_from_period(df["periodo"])

    residencia_df = standardize_residencia(df["residencia_raw"])
    df = pd.concat([df, residencia_df], axis=1)

    df["geo_level"] = np.where(df["nombre_provincia"].notna(), "provincia",
                        np.where(df["nombre_comunidad"].notna(), "comunidad", "nacional"))
    df["geo_name"] = np.select(
        [
            df["nombre_provincia"].notna(),
            df["nombre_comunidad"].notna()
        ],
        [
            df["nombre_provincia"],
            df["nombre_comunidad"]
        ],
        default="Total Nacional"
    )
    df["source_table"] = "df_comunidades"

    df = add_time_features(df, "fecha")

    return df


def quality_audit(df: pd.DataFrame, name: str):
    print(f"\n===== QUALITY AUDIT: {name} =====")
    print("Shape:", df.shape)
    print("\nNulls:")
    print(df.isna().sum().sort_values(ascending=False).head(20))
    print("\nDuplicate rows:", df.duplicated().sum())
    if "fecha" in df.columns:
        print("\nDate range:", df["fecha"].min(), "->", df["fecha"].max())
    if "measure_name" in df.columns:
        print("\nMeasures:", df["measure_name"].dropna().unique())
    if "geo_level" in df.columns:
        print("\nGeo levels:")
        print(df["geo_level"].value_counts(dropna=False))

=========================
### 4. KPI mart construction
=========================

In [26]:
def build_kpi_mart(
    df_long: pd.DataFrame,
    dims: list,
    value_col: str = "total"
) -> pd.DataFrame:
    kpi = (
        df_long
        .pivot_table(
            index=dims,
            columns="measure_name",
            values=value_col,
            aggfunc="sum"
        )
        .reset_index()
    )

    for c in ["Viajero", "Pernoctaciones"]:
        if c not in kpi.columns:
            kpi[c] = np.nan

    kpi = kpi.rename(columns={
        "Viajero": "viajeros",
        "Pernoctaciones": "pernoctaciones"
    })

    kpi["alos"] = safe_divide(kpi["pernoctaciones"], kpi["viajeros"])
    kpi = add_time_features(kpi, "fecha")

    return kpi


def add_growth_features(kpi: pd.DataFrame, group_cols: list) -> pd.DataFrame:
    out = kpi.sort_values(group_cols + ["fecha"]).copy()

    for metric in ["viajeros", "pernoctaciones", "alos"]:
        out[f"lag1_{metric}"] = out.groupby(group_cols)[metric].shift(1)
        out[f"lag12_{metric}"] = out.groupby(group_cols)[metric].shift(12)
        out[f"mom_{metric}"] = safe_divide(out[metric] - out[f"lag1_{metric}"], out[f"lag1_{metric}"]) * 100
        out[f"yoy_{metric}"] = safe_divide(out[metric] - out[f"lag12_{metric}"], out[f"lag12_{metric}"]) * 100
        out[f"roll3_{metric}"] = out.groupby(group_cols)[metric].transform(lambda s: s.rolling(3, min_periods=1).mean())
        out[f"roll12_{metric}"] = out.groupby(group_cols)[metric].transform(lambda s: s.rolling(12, min_periods=3).mean())

    return out


def add_shares(kpi: pd.DataFrame, geo_col="geo_name") -> pd.DataFrame:
    out = kpi.copy()
    monthly_totals = out.groupby("fecha")[["viajeros", "pernoctaciones"]].sum().rename(columns={
        "viajeros": "viajeros_total_mes",
        "pernoctaciones": "pernoctaciones_total_mes"
    }).reset_index()

    out = out.merge(monthly_totals, on="fecha", how="left")
    out["share_viajeros"] = safe_divide(out["viajeros"], out["viajeros_total_mes"])
    out["share_pernoctaciones"] = safe_divide(out["pernoctaciones"], out["pernoctaciones_total_mes"])
    return out

=========================
### 5. Seasonality
=========================

In [27]:
def add_seasonality_index(df: pd.DataFrame, group_cols: list, value_col: str) -> pd.DataFrame:
    out = df.copy()
    out["year_avg"] = out.groupby(group_cols + ["año"])[value_col].transform("mean")
    out[f"seasonality_idx_{value_col}"] = safe_divide(out[value_col], out["year_avg"]) * 100
    out.drop(columns=["year_avg"], inplace=True)
    return out


def seasonality_summary(df: pd.DataFrame, group_cols: list, value_col: str) -> pd.DataFrame:
    tmp = (
        df.groupby(group_cols + ["año", "mes"], as_index=False)[value_col]
          .sum()
    )

    summary = (
        tmp.groupby(group_cols + ["año"], as_index=False)[value_col]
           .agg(max_val="max", min_val="min", mean_val="mean", std_val="std")
           .reset_index(drop=True)
    )

    summary["peak_trough_ratio"] = safe_divide(summary["max_val"], summary["min_val"])
    summary["cv"] = safe_divide(summary["std_val"], summary["mean_val"])

    return summary


def compute_stl_components(series: pd.Series, period: int = 12):
    series = series.dropna().asfreq("MS")
    if len(series) < 24:
        return None
    stl = STL(series, period=period, robust=True)
    result = stl.fit()
    return result


=========================
### 6. Extreme cases and thresholds
=========================

In [28]:
def latest_snapshot(df: pd.DataFrame) -> pd.DataFrame:
    latest_date = df["fecha"].max()
    return df[df["fecha"] == latest_date].copy()


def latest_12m(df: pd.DataFrame, date_col="fecha") -> pd.DataFrame:
    max_date = df[date_col].max()
    start = max_date - pd.DateOffset(months=11)
    return df[(df[date_col] >= start) & (df[date_col] <= max_date)].copy()


def rank_top_bottom(df: pd.DataFrame, metric: str, n=10, ascending=False, min_volume_col=None, min_quantile=0.25):
    tmp = df.copy()
    if min_volume_col is not None:
        threshold = tmp[min_volume_col].quantile(min_quantile)
        tmp = tmp[tmp[min_volume_col] >= threshold].copy()

    tmp = tmp.sort_values(metric, ascending=ascending)
    top = tmp.head(n).copy()
    bottom = tmp.tail(n).copy() if ascending else tmp.tail(n).sort_values(metric, ascending=True).copy()
    return top, bottom


def add_statistical_flags(df: pd.DataFrame, peer_cols: list | None = None) -> pd.DataFrame:
    out = df.copy()
    if peer_cols is None or len(peer_cols) == 0:
        peer_cols = []

    metrics = [
        "alos",
        "yoy_viajeros",
        "yoy_pernoctaciones",
        "yoy_alos",
        "peak_trough_ratio",
        "cv"
    ]

    if len(peer_cols) == 0:
        # Global thresholds (no grouping)
        for metric in metrics:
            col = out[metric]
            out[f"p25_{metric}"] = col.quantile(0.25)
            out[f"p75_{metric}"] = col.quantile(0.75)
    else:
        # Per-peer thresholds
        for metric in metrics:
            out[f"p25_{metric}"] = out.groupby(peer_cols)[metric].transform(lambda s: s.quantile(0.25))
            out[f"p75_{metric}"] = out.groupby(peer_cols)[metric].transform(lambda s: s.quantile(0.75))

    out["flag_high_alos"] = out["alos"] > out["p75_alos"]
    out["flag_low_alos"] = out["alos"] < out["p25_alos"]
    out["flag_strong_yoy"] = out["yoy_pernoctaciones"] > out["p75_yoy_pernoctaciones"]
    out["flag_weak_yoy"] = out["yoy_pernoctaciones"] < out["p25_yoy_pernoctaciones"]
    out["flag_high_seasonality"] = (
        (out["peak_trough_ratio"] > out["p75_peak_trough_ratio"]) |
        (out["cv"] > out["p75_cv"])
    )

    return out

=========================
### 7. Forecasting functions
=========================

In [29]:
@dataclass
class ForecastResult:
    model_name: str
    geo_name: str
    segment_key: str
    metric_name: str
    fitted: pd.Series
    forecast: pd.Series
    lower: pd.Series
    upper: pd.Series
    scores: dict


def check_series_eligibility(series: pd.Series) -> dict:
    s = series.asfreq("MS")
    total_len = len(s)
    missing_pct = s.isna().mean()
    positive_obs = (s.fillna(0) > 0).sum()

    eligible = (
        total_len >= MIN_HISTORY and
        missing_pct <= MAX_MISSING_PCT and
        positive_obs >= MIN_POSITIVE_OBS
    )
    return {
        "eligible": eligible,
        "total_len": total_len,
        "missing_pct": missing_pct,
        "positive_obs": positive_obs
    }


def seasonal_naive_forecast(train: pd.Series, horizon: int = 12):
    train = train.asfreq("MS")
    last_year = train.iloc[-SEASONAL_PERIOD:].values
    fc_index = pd.date_range(train.index.max() + pd.offsets.MonthBegin(1), periods=horizon, freq="MS")
    reps = int(np.ceil(horizon / SEASONAL_PERIOD))
    pred = np.tile(last_year, reps)[:horizon]
    pred = pd.Series(pred, index=fc_index)

    resid = train.iloc[SEASONAL_PERIOD:] - train.shift(SEASONAL_PERIOD).iloc[SEASONAL_PERIOD:]
    sigma = np.nanstd(resid)
    lower = pred - 1.96 * sigma
    upper = pred + 1.96 * sigma
    return pred, lower, upper


def ets_forecast(train: pd.Series, horizon: int = 12):
    train = train.asfreq("MS")
    model = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=SEASONAL_PERIOD,
        initialization_method="estimated"
    )
    fit = model.fit(optimized=True)
    pred = fit.forecast(horizon)
    resid = train - fit.fittedvalues
    sigma = np.nanstd(resid)
    lower = pred - 1.96 * sigma
    upper = pred + 1.96 * sigma
    return fit.fittedvalues, pred, lower, upper


def sarima_forecast(train: pd.Series, horizon: int = 12, order=(1,1,1), seasonal_order=(1,1,1,12)):
    train = train.asfreq("MS")
    model = SARIMAX(
        train, 
        order=order, 
        seasonal_order=seasonal_order, 
        enforce_stationarity=False, 
        enforce_invertibility=False
    )
    fit = model.fit(disp=False)
    pred_obj = fit.get_forecast(steps=horizon)
    pred = pred_obj.predicted_mean
    conf = pred_obj.conf_int()
    lower = conf.iloc[:, 0]
    upper = conf.iloc[:, 1]
    fitted = fit.fittedvalues
    return fitted, pred, lower, upper


def rolling_backtest(series: pd.Series, model_name: str, horizon=12, min_train=36):
    s = series.dropna().asfreq("MS")
    if len(s) < min_train + horizon:
        return None

    train = s.iloc[:-horizon]
    test = s.iloc[-horizon:]

    try:
        if model_name == "seasonal_naive":
            pred, _, _ = seasonal_naive_forecast(train, horizon=horizon)
            fitted = pd.Series(index=train.index, dtype=float)
        elif model_name == "ets":
            fitted, pred, _, _ = ets_forecast(train, horizon=horizon)
        elif model_name == "sarima":
            fitted, pred, _, _ = sarima_forecast(train, horizon=horizon)
        else:
            return None

        pred = pred.reindex(test.index)

        scores = {
            "mae": mean_absolute_error(test, pred),
            "rmse": rmse(test, pred),
            "smape": smape(test, pred),
            "bias": float(np.mean(pred - test))
        }
        return scores
    except Exception:
        return None


def select_best_model(series: pd.Series):
    candidates = ["seasonal_naive", "ets", "sarima"]
    results = []

    for m in candidates:
        scores = rolling_backtest(series, m, horizon=min(12, max(6, len(series)//5)), min_train=36)
        if scores is not None:
            results.append((m, scores))

    if not results:
        return None, None

    ranking = sorted(results, key=lambda x: (x[1]["smape"], x[1]["rmse"]))
    best_model = ranking[0][0]
    score_table = pd.DataFrame([{**{"model": m}, **s} for m, s in results]).sort_values(["smape", "rmse"])
    return best_model, score_table


def forecast_one_series(series: pd.Series, geo_name: str, segment_key: str, metric_name: str) -> ForecastResult | None:
    eligibility = check_series_eligibility(series)
    if not eligibility["eligible"]:
        return None

    best_model, score_table = select_best_model(series.dropna())
    if best_model is None:
        return None

    train = series.dropna().asfreq("MS")

    if best_model == "seasonal_naive":
        pred, lower, upper = seasonal_naive_forecast(train, horizon=FORECAST_HORIZON)
        fitted = pd.Series(index=train.index, data=np.nan)
    elif best_model == "ets":
        fitted, pred, lower, upper = ets_forecast(train, horizon=FORECAST_HORIZON)
    else:
        fitted, pred, lower, upper = sarima_forecast(train, horizon=FORECAST_HORIZON)

    return ForecastResult(
        model_name=best_model,
        geo_name=geo_name,
        segment_key=segment_key,
        metric_name=metric_name,
        fitted=fitted,
        forecast=pred,
        lower=lower,
        upper=upper,
        scores=score_table.iloc[0].to_dict() if score_table is not None else {}
    )

=========================
### 8. Plotting helpers
=========================

In [30]:
def plot_bar_latest_12m(df, metric, title, color=COLOR_PRIMARY):
    agg = (
        latest_12m(df)
        .groupby("geo_name", as_index=False)[metric]
        .sum()
        .sort_values(metric, ascending=False)
    )

    fig = px.bar(
        agg,
        x=metric,
        y="geo_name",
        orientation="h",
        title=title,
        color_discrete_sequence=[color]
    )
    fig.update_layout(height=600, yaxis_title="", xaxis_title=metric)
    return fig


def plot_line_trend(df: pd.DataFrame, geo_name: str, metric: str, title: str):
    tmp = (
        df[df["geo_name"] == geo_name]
        .copy()
        .sort_values("fecha")
    )

    tmp = tmp[tmp[metric].notna()].copy()

    print(f"\n=== plot_line_trend debug: {geo_name} / {metric} ===")
    print("Shape:", tmp.shape)
    print(tmp[["fecha", "geo_name", metric]].tail(6))

    fig = px.line(
        tmp,
        x="fecha",
        y=metric,
        markers=True,
        title=title
    )

    fig.update_layout(
        height=450,
        xaxis_title="Fecha",
        yaxis_title=metric.capitalize(),
        hovermode="x unified"
    )

    fig.update_traces(mode="lines+markers")

    return fig


def plot_seasonality_heatmap(df, metric, title):
    tmp = (
        df.groupby(["geo_name", "mes"], as_index=False)[metric]
            .mean()
            .pivot(index="geo_name", columns="mes", values=metric)
            .reindex(TARGET_GEOS)
    )

    fig = px.imshow(
        tmp,
        aspect="auto",
        color_continuous_scale="Tealgrn",
        labels=dict(x="Mes", y="Geografía", color=metric),
        title=title
    )
    fig.update_layout(height=500)
    return fig


def plot_scatter_growth_vs_alos(df, title):
    tmp = latest_snapshot(df).copy()
    tmp = tmp[tmp["geo_name"].isin(TARGET_GEOS)]

    fig = px.scatter(
        tmp,
        x="yoy_viajeros",
        y="alos",
        size="pernoctaciones",
        color="geo_name",
        hover_data=["viajeros", "pernoctaciones", "yoy_pernoctaciones", "yoy_alos"],
        title=title
    )
    fig.add_hline(y=tmp["alos"].median(), line_dash="dash", line_color="gray")
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(height=600)
    return fig


def plot_forecast(actual_series, forecast_result: ForecastResult, title: str):
    hist = actual_series.dropna().asfreq("MS")
    fc = forecast_result.forecast
    lower = forecast_result.lower
    upper = forecast_result.upper

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hist.index, y=hist.values,
        mode="lines", name="Actual",
        line=dict(color=COLOR_PRIMARY, width=3)
    ))
    fig.add_trace(go.Scatter(
        x=fc.index, y=fc.values,
        mode="lines", name=f"Forecast ({forecast_result.model_name})",
        line=dict(color=COLOR_ACCENT, width=3, dash="dash")
    ))
    fig.add_trace(go.Scatter(
        x=list(fc.index) + list(fc.index[::-1]),
        y=list(upper.values) + list(lower.values[::-1]),
        fill="toself",
        fillcolor="rgba(217,122,0,0.20)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=True,
        name="95% interval"
    ))
    fig.update_layout(title=title, height=500)
    return fig

=========================
### 9. Bottleneck and opportunity logic
=========================

In [31]:
def build_business_scores(df_latest: pd.DataFrame) -> pd.DataFrame:
    out = df_latest.copy()

    for c in ["yoy_viajeros", "yoy_pernoctaciones", "alos"]:
        out[f"rz_{c}"] = robust_zscore(winsorize_series(out[c].astype(float)))

    if "peak_trough_ratio" in out.columns:
        out["rz_peak_trough_ratio"] = robust_zscore(winsorize_series(out["peak_trough_ratio"].astype(float)))
    else:
        out["rz_peak_trough_ratio"] = 0.0

    if "forecast_downside_risk" not in out.columns:
        out["forecast_downside_risk"] = 0.0
    out["rz_forecast_downside_risk"] = robust_zscore(winsorize_series(out["forecast_downside_risk"].astype(float)))

    out["business_score"] = (
        0.30 * out["rz_yoy_pernoctaciones"] +
        0.20 * out["rz_yoy_viajeros"] +
        0.20 * out["rz_alos"] -
        0.20 * out["rz_peak_trough_ratio"] -
        0.10 * out["rz_forecast_downside_risk"]
    )

    conditions = [
        (out["yoy_viajeros"] < 0) & (out["alos"] >= out["alos"].quantile(0.75)),
        (out["yoy_viajeros"] > 0) & (out["yoy_alos"] < 0),
        (out["peak_trough_ratio"] >= out["peak_trough_ratio"].quantile(0.75)),
        (out["yoy_viajeros"] > 0) & (out["yoy_pernoctaciones"] > 0) & (out["alos"] >= out["alos"].quantile(0.75))
    ]
    labels = [
        "Acquisition gap",
        "Stay-depth gap",
        "Seasonality stress",
        "Premium opportunity"
    ]
    out["business_flag"] = np.select(conditions, labels, default="Stable / mixed")

    out["score_band"] = pd.qcut(
        out["business_score"].rank(method="first"),
        q=4,
        labels=["Critical", "Watch", "Healthy", "Opportunity"]
    )

    return out

=========================
### 10. Main pipeline
=========================

In [32]:
# Prepare sources
aloj = prepare_df_alojamientos(df_alojamientos)
geo = prepare_df_comunidades(df_comunidades)

quality_audit(aloj, "Alojamientos")
quality_audit(geo, "Comunidades")

# Filter history window
aloj = aloj[(aloj["año"] >= 2021) & (aloj["año"] <= 2025)].copy()
geo = geo[(geo["año"] >= 2021) & (geo["año"] <= 2025)].copy()

# Flag focus geographies
aloj["focus_flag"] = aloj["geo_name"].isin(TARGET_GEOS)
geo["focus_flag"] = geo["geo_name"].isin(TARGET_GEOS)

# Main KPI marts
dims_aloj = [
    "fecha", "geo_level", "geo_name", "nombre_comunidad",
    "tipo_alojamiento", "residencia_group", "residencia_detail"
]
kpi_aloj = build_kpi_mart(aloj, dims=dims_aloj)

dims_geo = [
    "fecha", "geo_level", "geo_name", "nombre_comunidad", "nombre_provincia",
    "residencia_group", "residencia_detail"
]
kpi_geo = build_kpi_mart(geo, dims=dims_geo)

# Add growth features
kpi_aloj = add_growth_features(
    kpi_aloj,
    group_cols=["geo_name", "tipo_alojamiento", "residencia_group", "residencia_detail"]
)
kpi_geo = add_growth_features(
    kpi_geo,
    group_cols=["geo_name", "residencia_group", "residencia_detail"]
)

# Add shares
kpi_aloj = add_shares(kpi_aloj)
kpi_geo = add_shares(kpi_geo)

# Add seasonality indices
for metric in ["viajeros", "pernoctaciones", "alos"]:
    kpi_aloj = add_seasonality_index(kpi_aloj, ["geo_name", "tipo_alojamiento", "residencia_group"], metric)
    kpi_geo = add_seasonality_index(kpi_geo, ["geo_name", "residencia_group"], metric)

# Seasonality summaries
season_geo = seasonality_summary(kpi_geo, ["geo_name"], "pernoctaciones")
season_aloj = seasonality_summary(kpi_aloj, ["geo_name", "tipo_alojamiento"], "pernoctaciones")

# Attach latest annual seasonality summary back to KPI snapshots
season_geo_latest = season_geo[season_geo["año"] == season_geo["año"].max()].copy()

latest_geo = latest_snapshot(kpi_geo)
latest_geo = latest_geo.merge(
    season_geo_latest[["geo_name", "peak_trough_ratio", "cv"]],
    on="geo_name", how="left"
)

latest_geo = add_statistical_flags(latest_geo, peer_cols=[])
latest_geo_focus = latest_geo[latest_geo["geo_name"].isin(TARGET_GEOS)].copy()

# Build business scoring table
score_table = build_business_scores(latest_geo_focus)


===== QUALITY AUDIT: Alojamientos =====
Shape: (36000, 17)

Nulls:
total                4236
nombre_comunidad     1800
tipo_alojamiento        0
measure_name            0
residencia_raw          0
periodo                 0
año                     0
mes                     0
fecha                   0
residencia_detail       0
residencia_group        0
geo_level               0
geo_name                0
source_table            0
quarter                 0
month_name              0
month_id                0
dtype: int64

Duplicate rows: 0

Date range: 2021-01-01 00:00:00 -> 2025-12-01 00:00:00

Measures: <StringArray>
['Viajero', 'Pernoctaciones']
Length: 2, dtype: string

Geo levels:
geo_level
comunidad    34200
nacional      1800
Name: count, dtype: int64

===== QUALITY AUDIT: Comunidades =====
Shape: (25200, 17)

Nulls:
nombre_provincia     7200
nombre_comunidad      360
total                   1
measure_name            0
residencia_raw          0
periodo                 0
año         

=========================
### 11. Forecasting execution
=========================

In [33]:
kpi_geo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 40 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   fecha                           9000 non-null   datetime64[ns]
 1   geo_level                       9000 non-null   object        
 2   geo_name                        9000 non-null   object        
 3   nombre_comunidad                9000 non-null   string        
 4   nombre_provincia                9000 non-null   string        
 5   residencia_group                9000 non-null   string        
 6   residencia_detail               9000 non-null   string        
 7   pernoctaciones                  9000 non-null   float64       
 8   viajeros                        9000 non-null   float64       
 9   alos                            8999 non-null   float64       
 10  año                             9000 non-null   int32         
 11  mes 

In [ ]:
forecast_records = []
forecast_plot_objects = []

base_forecast_input = (
    kpi_geo[kpi_geo["geo_name"].isin(TARGET_GEOS)]
    .groupby(["fecha", "geo_name"], as_index=False)[["viajeros", "pernoctaciones"]]
    .sum()
)

for geo_name in TARGET_GEOS:
    tmp = base_forecast_input[base_forecast_input["geo_name"] == geo_name].sort_values("fecha").copy()
    tmp = tmp.set_index("fecha").asfreq("MS")

    for metric in ["viajeros", "pernoctaciones"]:
        series = tmp[metric]
        fr = forecast_one_series(
            series=series,
            geo_name=geo_name,
            segment_key="TOTAL",
            metric_name=metric
        )

        if fr is not None:
            fc_df = pd.DataFrame({
                "fecha": fr.forecast.index,
                "geo_name": geo_name,
                "segment_key": "TOTAL",
                "metric_name": metric,
                "forecast": fr.forecast.values,
                "lower": fr.lower.values,
                "upper": fr.upper.values,
                "model_name": fr.model_name
            })
            forecast_records.append(fc_df)

            fig = plot_forecast(
                actual_series=series,
                forecast_result=fr,
                title=f"{geo_name} - {metric} forecast"
            )
            forecast_plot_objects.append((f"{geo_name}_{metric}_forecast.html", fig))

forecast_output = pd.concat(forecast_records, ignore_index=True) if forecast_records else pd.DataFrame()

# Pivot forecast to derive ALOS forecast
if not forecast_output.empty:
    fc_wide = (
        forecast_output
        .pivot_table(
            index=["fecha", "geo_name", "segment_key"],
            columns="metric_name",
            values="forecast",
            aggfunc="sum"
        )
        .reset_index()
    )
    fc_wide["alos_forecast"] = safe_divide(fc_wide["pernoctaciones"], fc_wide["viajeros"])
else:
    fc_wide = pd.DataFrame()

# Add simple forecast downside risk
# Only try to add forecast_downside_risk if we have forecast data
if not forecast_output.empty:
    latest_hist = (
        base_forecast_input
        .sort_values("fecha")
        .groupby("geo_name", as_index=False)
        .tail(1)[["geo_name", "viajeros", "pernoctaciones"]]
        .rename(columns={
            "viajeros": "latest_viajeros",
            "pernoctaciones": "latest_pernoctaciones"
        })
    )

    fc_risk = (
        forecast_output.groupby(["geo_name", "metric_name"], as_index=False)
        .agg(avg_forecast=("forecast", "mean"), avg_lower=("lower", "mean"))
    )

    fc_risk = fc_risk.merge(latest_hist, on="geo_name", how="left")
    fc_risk["forecast_downside_risk"] = np.where(
        ((fc_risk["metric_name"] == "viajeros") & (fc_risk["avg_lower"] < fc_risk["latest_viajeros"])) |
        ((fc_risk["metric_name"] == "pernoctaciones") & (fc_risk["avg_lower"] < fc_risk["latest_pernoctaciones"])),
        1, 0
    )

    downside_summary = fc_risk.groupby("geo_name", as_index=False)["forecast_downside_risk"].max()

    # Merge downside_summary into score_table
    score_table = score_table.drop(columns=["forecast_downside_risk"], errors="ignore")
    score_table = score_table.merge(downside_summary, on="geo_name", how="left")
else:
    # No forecast data: create a zero column
    score_table = score_table.drop(columns=["forecast_downside_risk"], errors="ignore")
    score_table["forecast_downside_risk"] = 0.0


score_table["forecast_downside_risk"] = score_table["forecast_downside_risk"].fillna(0)
score_table = build_business_scores(score_table)

In [35]:
score_table

,fecha,geo_level,geo_name,nombre_comunidad,nombre_provincia,residencia_group,residencia_detail,pernoctaciones,viajeros,alos,año,mes,quarter,month_name,month_id,lag1_viajeros,lag12_viajeros,mom_viajeros,yoy_viajeros,roll3_viajeros,roll12_viajeros,lag1_pernoctaciones,lag12_pernoctaciones,mom_pernoctaciones,yoy_pernoctaciones,roll3_pernoctaciones,roll12_pernoctaciones,lag1_alos,lag12_alos,mom_alos,yoy_alos,roll3_alos,roll12_alos,viajeros_total_mes,pernoctaciones_total_mes,share_viajeros,share_pernoctaciones,seasonality_idx_viajeros,seasonality_idx_pernoctaciones,seasonality_idx_alos,peak_trough_ratio,cv,p25_alos,p75_alos,p25_yoy_viajeros,p75_yoy_viajeros,p25_yoy_pernoctaciones,p75_yoy_pernoctaciones,p25_yoy_alos,p75_yoy_alos,p25_peak_trough_ratio,p75_peak_trough_ratio,p25_cv,p75_cv,flag_high_alos,flag_low_alos,flag_strong_yoy,flag_weak_yoy,flag_high_seasonality,rz_yoy_viajeros,rz_yoy_pernoctaciones,rz_alos,rz_peak_trough_ratio,rz_forecast_downside_risk,business_score,business_flag,score_band,forecast_downside_risk
0,2025-12-01,provincia,"Balears, Illes","Balears, Illes","Balears, Illes",Detalle,Total,323934.0,104589.0,3.097209,2025,12,4,December,202512,139514.0,102526.0,-25.033330,2.012173,5.141913e+05,1.034753e+06,435053.0,311450.0,-25.541486,4.008348,2.575444e+06,5.289958e+06,3.118347,3.037766,-0.677842,1.956801,3.860454,4.468690,13856788.0,37056988.0,0.007548,0.008742,10.107633,6.123565,69.309112,47.654943,0.868769,1.687876,2.089925,0.137335,11.1013,-1.99527,11.173948,-5.22088,1.931372,2.156457,4.023363,0.21038,0.380598,True,False,False,False,True,-0.674500,-0.401443,2.366742,32.359951,0.0,-6.253975,Seasonality stress,Critical,0
1,2025-12-01,provincia,"Balears, Illes","Balears, Illes","Balears, Illes",Doméstico,Residentes en España,75792.0,38624.0,1.962303,2025,12,4,December,202512,51612.0,39500.0,-25.164690,-2.217722,7.782267e+04,1.235437e+05,94443.0,84821.0,-19.748420,-10.644770,1.967423e+05,4.333791e+05,1.829865,2.147367,7.237587,-8.618175,2.241473,3.162674,13856788.0,37056988.0,0.002787,0.002045,31.263440,17.488615,62.045707,47.654943,0.868769,1.687876,2.089925,0.137335,11.1013,-1.99527,11.173948,-5.22088,1.931372,2.156457,4.023363,0.21038,0.380598,False,False,False,True,True,-1.572967,-2.948329,-0.203387,32.359951,0.0,-7.711760,Seasonality stress,Critical,0
2,2025-12-01,provincia,"Balears, Illes","Balears, Illes","Balears, Illes",Internacional,Residentes en el Extranjero,248142.0,65965.0,3.761722,2025,12,4,December,202512,87902.0,63026.0,-24.956201,4.663155,4.363687e+05,9.112091e+05,340609.0,226629.0,-27.147550,9.492607,2.378701e+06,4.856578e+06,3.874872,3.595802,-2.920094,4.614281,4.434708,4.828480,13856788.0,37056988.0,0.004760,0.006696,7.239283,5.109400,77.906968,47.654943,0.868769,1.687876,2.089925,0.137335,11.1013,-1.99527,11.173948,-5.22088,1.931372,2.156457,4.023363,0.21038,0.380598,True,False,False,False,True,-0.096617,0.555558,3.798540,32.359951,0.0,-5.564938,Seasonality stress,Critical,0
3,2025-12-01,provincia,Barcelona,Cataluña,Barcelona,Detalle,Total,2020125.0,917329.0,2.202182,2025,12,4,December,202512,995175.0,830151.0,-7.822343,10.501463,1.090441e+06,1.156386e+06,2213803.0,1900241.0,-8.748656,6.308884,2.502624e+06,2.887473e+06,2.224536,2.289031,-1.004921,-3.794139,2.278707,2.464465,13856788.0,37056988.0,0.066201,0.054514,79.327197,69.961701,89.357386,2.218917,0.271844,1.687876,2.089925,0.137335,11.1013,-1.99527,11.173948,-5.22088,1.931372,2.156457,4.023363,0.21038,0.380598,True,False,False,False,False,1.176064,0.000000,0.339846,-0.066059,0.0,0.316394,Stay-depth gap,Healthy,1
4,2025-12-01,provincia,Barcelona,Cataluña,Barcelona,Doméstico,Residentes en España,496206.0,290040.0,1.710819,2025,12,4,December,202512,288876.0,275949.0,0.402941,5.106378,3.096760e+05,2.914722e+05,495002.0,485475.0,0.243231,2.210412,5.288173e+05,5.405072e+05,1.713545,1.759292,-0.159069,-2.755271,1.708172,1.853473,13856788.0,37056988.0,0.020931,0.013390,99.508615,91.803764,92.303423,2.218917,0.271844,1.687876,2.089925,0.137335,11.1013,-

=========================
### 12. Executive output tables
=========================

In [36]:
tbl_exec_latest_12m = (
    latest_12m(kpi_geo[kpi_geo["geo_name"].isin(TARGET_GEOS)])
    .groupby("geo_name", as_index=False)[["viajeros", "pernoctaciones"]]
    .sum()
)
tbl_exec_latest_12m["alos"] = safe_divide(tbl_exec_latest_12m["pernoctaciones"], tbl_exec_latest_12m["viajeros"])

tbl_exec_yoy = (
    latest_snapshot(kpi_geo[kpi_geo["geo_name"].isin(TARGET_GEOS)])
    [["geo_name", "viajeros", "pernoctaciones", "alos", "yoy_viajeros", "yoy_pernoctaciones", "yoy_alos"]]
    .sort_values("pernoctaciones", ascending=False)
)

tbl_seasonality_summary = (
    season_geo_latest[season_geo_latest["geo_name"].isin(TARGET_GEOS)]
    .sort_values("peak_trough_ratio", ascending=False)
)

tbl_bottlenecks = score_table.sort_values("business_score")
tbl_opportunities = score_table.sort_values("business_score", ascending=False)

# Extreme cases by geography
latest_geo_focus = latest_snapshot(kpi_geo)
latest_geo_focus = latest_geo_focus[latest_geo_focus["geo_name"].isin(TARGET_GEOS)]

top_viajeros, bottom_viajeros = rank_top_bottom(
    latest_geo_focus,
    metric="viajeros",
    n=len(TARGET_GEOS),
    ascending=False
)

top_alos, bottom_alos = rank_top_bottom(
    latest_geo_focus,
    metric="alos",
    n=len(TARGET_GEOS),
    ascending=False,
    min_volume_col="viajeros",
    min_quantile=0.25
)

tbl_extremes_top_bottom = {
    "top_viajeros": top_viajeros,
    "bottom_viajeros": bottom_viajeros,
    "top_alos": top_alos,
    "bottom_alos": bottom_alos
}

# Segment detail at accommodation-type level
tbl_segment_performance_detail = (
    latest_snapshot(kpi_aloj)
    .query("geo_name in @TARGET_GEOS")
    [["geo_name", "tipo_alojamiento", "residencia_group", "viajeros", "pernoctaciones", "alos",
      "yoy_viajeros", "yoy_pernoctaciones", "yoy_alos", "share_viajeros", "share_pernoctaciones"]]
    .sort_values(["geo_name", "pernoctaciones"], ascending=[True, False])
)

In [37]:
tbl_segment_performance_detail

measure_name,geo_name,tipo_alojamiento,residencia_group,viajeros,pernoctaciones,alos,yoy_viajeros,yoy_pernoctaciones,yoy_alos,share_viajeros,share_pernoctaciones
2759,"Balears, Illes",Encuesta de Ocupación Hotelera,Detalle,104589.0,323934.0,3.097209,2.012173,4.008348,1.956801,0.006065,0.006283
2879,"Balears, Illes",Encuesta de Ocupación Hotelera,Internacional,65965.0,248142.0,3.761722,4.663155,9.492607,4.614281,0.003825,0.004813
2819,"Balears, Illes",Encuesta de Ocupación Hotelera,Doméstico,38624.0,75792.0,1.962303,-2.217722,-10.644770,-8.618175,0.002240,0.001470
3299,"Balears, Illes",Encuesta de Ocupación en Apartamentos Turísticos,Detalle,7542.0,40711.0,5.397905,4.474304,5.891380,1.356387,0.000437,0.000790
3419,"Balears, Illes",Encuesta de Ocupación en Apartamentos Turísticos,Internacional,6729.0,37994.0,5.646307,33.857171,21.173657,-9.475409,0.000390,0.000737
3119,"Balears, Illes",Encuesta de Ocupación en Alojamientos de Turis...,Detalle,10347.0,31170.0,3.012467,-21.572046,-0.928104,26.322173,0.000600,0.000605
3239,"Balears, Illes",Encuesta de Ocupación en Alojamientos de Turis...,Internacional,6059.0,22365.0,3.691203,-17.944204,9.594747,33.561250,0.000351,0.000434
3179,"Balears, Illes",Encuesta de Ocupación en Alojamientos de Turis...,Doméstico,4288.0,8805.0,2.053405,-26.183508,-20.352782,7.898949,0.000249,0.000171
3359,"Balears, Illes",Encuesta de Ocupación en Apartamentos Turísticos,Doméstico,813.0,2717.0,3.341943,-62.927497,-61.683825,3.354702,0.000047,0.000053
2939,"Balears, Illes",Encuesta de Ocupación en Albergues,Detalle,0.0,0.0,NaN,NaN,NaN,NaN,0.000000,0.000000


=========================
### 13. Plot generation
=========================

In [38]:
# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================
pio.renderers.default = "vscode"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLOR_PRIMARIO = "#1f77b4"
COLOR_SECUNDARIO = "#ff7f0e"
COLOR_TERCIARIO = "#2ca02c"
COLOR_CUARTO = "#d62728"
COLOR_GRIS = "#7f7f7f"

MAPA_COLORES = {
    "viajeros": COLOR_PRIMARIO,
    "pernoctaciones": COLOR_SECUNDARIO,
    "alos": COLOR_TERCIARIO,
}

NOMBRES_METRICAS = {
    "viajeros": "Viajeros",
    "pernoctaciones": "Pernoctaciones",
    "alos": "Estancia media",
    "yoy_pernoctaciones": "Variación interanual de pernoctaciones (%)",
}

NOTAS_ABREVIATURAS = [
    "Nota: ALOS = estancia media (Average Length of Stay).",
    "Nota: YoY = variación interanual.",
    "Nota: OnS = pernoctaciones."
]


# =========================================================
# UTILIDADES
# =========================================================
def aplicar_layout_estandar(fig, height=600, titulo=None):
    fig.update_layout(
        height=height,
        template="plotly_white",
        hovermode="closest",
        margin=dict(l=40, r=30, t=80, b=60),
        font=dict(size=12),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        )
    )
    if titulo is not None:
        fig.update_layout(title=titulo)
    return fig


def safe_divide(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return np.where((b.notna()) & (b != 0), a / b, np.nan)


def add_time_features(df, fecha_col="fecha"):
    df = df.copy()
    df[fecha_col] = pd.to_datetime(df[fecha_col], errors="coerce")
    df["año"] = df[fecha_col].dt.year
    df["mes"] = df[fecha_col].dt.month
    df["mes_nombre"] = df[fecha_col].dt.month_name()
    return df


def add_growth_features(df, group_cols, value_col="pernoctaciones", fecha_col="fecha"):
    df = df.copy().sort_values(group_cols + [fecha_col])
    lag_col = f"lag_{value_col}"
    yoy_col = f"yoy_{value_col}"

    df[lag_col] = df.groupby(group_cols)[value_col].shift(12)
    df[yoy_col] = np.where(
        (df[lag_col].notna()) & (df[lag_col] != 0),
        (df[value_col] / df[lag_col] - 1) * 100,
        np.nan
    )
    return df


def latest_12m(df, fecha_col="fecha"):
    df = df.copy()
    df[fecha_col] = pd.to_datetime(df[fecha_col], errors="coerce")
    max_fecha = df[fecha_col].max()
    if pd.isna(max_fecha):
        return df.iloc[0:0].copy()
    cutoff = max_fecha - pd.DateOffset(months=11)
    return df[df[fecha_col].between(cutoff, max_fecha)].copy()


def traducir_mes_num_a_es():
    return {
        1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
        7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"
    }


def ordenar_geos_por_12m(df, metric):
    base_12m = (
        latest_12m(df)
        .groupby("geo_name", as_index=False)[metric]
        .sum()
        .sort_values(metric, ascending=False)
    )
    return base_12m["geo_name"].tolist()


def agregar_nota_abreviaturas(fig, texto_extra=None, yshift=-0.20):
    texto = "<br>".join(NOTAS_ABREVIATURAS if texto_extra is None else NOTAS_ABREVIATURAS + [texto_extra])
    fig.add_annotation(
        x=0,
        y=yshift,
        xref="paper",
        yref="paper",
        text=texto,
        showarrow=False,
        align="left",
        font=dict(size=10, color="gray")
    )
    return fig


def guardar_html(fig, nombre_archivo):
    ruta = OUTPUT_DIR / nombre_archivo
    fig.write_html(ruta, include_plotlyjs="cdn")
    return ruta


def print_tabla_titulo(titulo, df):
    print("\n" + "=" * 100)
    print(titulo)
    print("=" * 100)
    print(df.to_string(index=False))


# =========================================================
# CONSTRUCCIÓN DE MARTS
# =========================================================
def build_geo_month_base(kpi_geo, target_geos):
    df = (
        kpi_geo[kpi_geo["geo_name"].isin(target_geos)]
        .groupby(["fecha", "geo_name"], as_index=False)
        .agg(
            viajeros=("viajeros", "sum"),
            pernoctaciones=("pernoctaciones", "sum")
        )
    )

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["viajeros"] = pd.to_numeric(df["viajeros"], errors="coerce")
    df["pernoctaciones"] = pd.to_numeric(df["pernoctaciones"], errors="coerce")
    df = df.dropna(subset=["fecha", "geo_name", "viajeros", "pernoctaciones"]).copy()

    df["alos"] = safe_divide(df["pernoctaciones"], df["viajeros"])
    df = add_time_features(df, "fecha")
    df = add_growth_features(df, group_cols=["geo_name"], value_col="pernoctaciones", fecha_col="fecha")
    return df


def build_geo_year_base(kpi_geo_plot, years=(2021, 2022, 2023, 2024, 2025)):
    df_year = (
        kpi_geo_plot[kpi_geo_plot["año"].isin(years)]
        .groupby(["año", "geo_name"], as_index=False)
        .agg(
            viajeros=("viajeros", "sum"),
            pernoctaciones=("pernoctaciones", "sum")
        )
        .sort_values(["geo_name", "año"])
    )

    df_year["alos"] = safe_divide(df_year["pernoctaciones"], df_year["viajeros"])
    df_year["lag_pernoctaciones_year"] = df_year.groupby("geo_name")["pernoctaciones"].shift(1)
    df_year["yoy_pernoctaciones"] = np.where(
        (df_year["lag_pernoctaciones_year"].notna()) & (df_year["lag_pernoctaciones_year"] != 0),
        (df_year["pernoctaciones"] / df_year["lag_pernoctaciones_year"] - 1) * 100,
        np.nan
    )
    return df_year


def build_seasonality_base(kpi_geo_plot):
    geo_month_avg = (
        kpi_geo_plot
        .groupby(["geo_name", "mes"], as_index=False)
        .agg(
            avg_pernoctaciones=("pernoctaciones", "mean"),
            avg_alos=("alos", "mean")
        )
    )
    return geo_month_avg


def build_accommodation_base(kpi_aloj, focus_communities, years=(2021, 2022, 2023, 2024, 2025)):
    df = kpi_aloj.copy()
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df = add_time_features(df, "fecha")
    df = df[
        df["geo_name"].isin(focus_communities) &
        df["año"].isin(years)
    ].copy()

    for col in ["viajeros", "pernoctaciones"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["geo_name", "tipo_alojamiento", "fecha", "viajeros", "pernoctaciones"]).copy()
    df["alos"] = safe_divide(df["pernoctaciones"], df["viajeros"])
    return df


# =========================================================
# 1) BARRAS 5 AÑOS POR GEOGRAFÍA
# =========================================================
def plot_barras_5_anios_por_geo(df_year, orden_geos, metric, title):
    tmp = df_year.copy()
    tmp["geo_name"] = pd.Categorical(tmp["geo_name"], categories=orden_geos, ordered=True)
    tmp = tmp.sort_values(["geo_name", "año"])

    fig = px.bar(
        tmp,
        x="geo_name",
        y=metric,
        color="año",
        barmode="group",
        text=metric,
        title=title,
        category_orders={"geo_name": orden_geos}
    )
    fig.update_traces(texttemplate="%{text:,.0f}" if metric != "alos" else "%{text:.2f}", textposition="outside")
    fig.update_xaxes(title="")
    fig.update_yaxes(title=NOMBRES_METRICAS.get(metric, metric), separatethousands=True)
    aplicar_layout_estandar(fig, height=650)
    agregar_nota_abreviaturas(fig, yshift=-0.24)
    return fig


# =========================================================
# 2) TRAYECTORIA ANUAL 2023-2025 + TABLA 2022 VS 2023-2025
# =========================================================
def make_tabla_resumen_yoy_2022_vs_2023_2025(df_year):
    yoy_2022 = (
        df_year[df_year["año"] == 2022][["geo_name", "yoy_pernoctaciones"]]
        .rename(columns={"yoy_pernoctaciones": "valor"})
        .assign(periodo="2022")
    )

    yoy_avg_2325 = (
        df_year[df_year["año"].between(2023, 2025)]
        .groupby("geo_name", as_index=False)["yoy_pernoctaciones"]
        .mean()
        .rename(columns={"yoy_pernoctaciones": "valor"})
        .assign(periodo="2023-2025")
    )

    tabla = pd.concat([yoy_2022, yoy_avg_2325], ignore_index=True)
    tabla = tabla.rename(columns={"geo_name": "ubicacion"})
    tabla["valor"] = pd.to_numeric(tabla["valor"], errors="coerce").round(2)
    tabla = tabla[["periodo", "ubicacion", "valor"]].sort_values(["ubicacion", "periodo"])
    return tabla


def plot_trayectoria_alos_yoy_con_tabla(df_year, years_plot=(2023, 2024, 2025), yoy_cap=200):
    plot_df = df_year[df_year["año"].isin(years_plot)].copy()
    plot_df = plot_df.dropna(subset=["yoy_pernoctaciones", "alos"]).copy()

    plot_df["yoy_plot"] = plot_df["yoy_pernoctaciones"].clip(-yoy_cap, yoy_cap)
    plot_df["nota_eje"] = np.where(
        plot_df["yoy_pernoctaciones"].abs() > yoy_cap,
        "Valor recortado para visualización",
        "Valor dentro del rango del eje"
    )

    tabla = make_tabla_resumen_yoy_2022_vs_2023_2025(df_year)

    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.68, 0.32],
        specs=[[{"type": "scatter"}, {"type": "table"}]],
        horizontal_spacing=0.06,
        subplot_titles=(
            "Trayectoria anual de crecimiento vs estancia media (2023-2025)",
            "Resumen del efecto 2022 y promedio 2023-2025"
        )
    )

    for geo in sorted(plot_df["geo_name"].unique()):
        tmp = plot_df[plot_df["geo_name"] == geo].sort_values("año")
        fig.add_trace(
            go.Scatter(
                x=tmp["yoy_plot"],
                y=tmp["alos"],
                mode="lines+markers+text",
                name=geo,
                text=tmp["año"].astype(str),
                textposition="top center",
                customdata=np.stack(
                    [
                        tmp["geo_name"],
                        tmp["año"].astype(str),
                        tmp["viajeros"].round(0),
                        tmp["pernoctaciones"].round(0),
                        tmp["yoy_pernoctaciones"].round(2),
                        tmp["nota_eje"]
                    ],
                    axis=-1
                ),
                hovertemplate=(
                    "Ubicación: %{customdata[0]}<br>"
                    "Año: %{customdata[1]}<br>"
                    "Viajeros: %{customdata[2]:,.0f}<br>"
                    "Pernoctaciones: %{customdata[3]:,.0f}<br>"
                    "Variación interanual real: %{customdata[4]:.2f}%<br>"
                    "Estancia media: %{y:.2f}<br>"
                    "Nota del eje: %{customdata[5]}<extra></extra>"
                )
            ),
            row=1, col=1
        )

    fig.add_trace(
        go.Table(
            header=dict(
                values=["Periodo", "Ubicación", "Valor"],
                fill_color="#E8EEF7",
                align="left"
            ),
            cells=dict(
                values=[
                    tabla["periodo"],
                    tabla["ubicacion"],
                    tabla["valor"].map(lambda x: f"{x:,.2f}" if pd.notna(x) else "")
                ],
                align="left"
            )
        ),
        row=1, col=2
    )

    fig.update_xaxes(
        title_text=f"Variación interanual de pernoctaciones (%) - recorte visual ±{yoy_cap}",
        row=1, col=1
    )
    fig.update_yaxes(title_text="Estancia media (ALOS)", row=1, col=1)

    fig.update_layout(
        template="plotly_white",
        height=700,
        margin=dict(l=40, r=30, t=90, b=80),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        ),
        title="Trayectoria anual de crecimiento y estancia media por ubicación"
    )

    agregar_nota_abreviaturas(
        fig,
        texto_extra="Nota metodológica: el gráfico omite 2022 para reducir el efecto rebote post-COVID, pero la tabla mantiene 2022 como referencia.",
        yshift=-0.18
    )

    return fig, tabla


# =========================================================
# 3) JOINTPLOTS MENSUALES POR PARES DE UBICACIONES
# =========================================================
def preparar_pares_mensuales(df_month, years=(2021, 2022, 2023, 2024, 2025)):
    tmp = df_month[df_month["año"].isin(years)].copy()

    pares = []
    geos = sorted(tmp["geo_name"].unique())
    for g1, g2 in combinations(geos, 2):
        base = tmp[tmp["geo_name"].isin([g1, g2])].copy()
        pivot = (
            base.pivot_table(
                index=["fecha", "año", "mes"],
                columns="geo_name",
                values=["viajeros", "pernoctaciones", "alos"],
                aggfunc="sum"
            )
        )
        pivot.columns = [f"{m}__{g}" for m, g in pivot.columns]
        pivot = pivot.reset_index()

        for metric in ["viajeros", "pernoctaciones", "alos"]:
            c1 = f"{metric}__{g1}"
            c2 = f"{metric}__{g2}"
            if c1 in pivot.columns and c2 in pivot.columns:
                pares.append((g1, g2, metric, pivot[["fecha", "año", "mes", c1, c2]].copy()))
    return pares


def detectar_anomalias_iqr(df, xcol, ycol):
    tmp = df[[xcol, ycol]].dropna().copy()
    if tmp.empty:
        return pd.Series(False, index=df.index)

    x_q1, x_q3 = tmp[xcol].quantile([0.25, 0.75])
    y_q1, y_q3 = tmp[ycol].quantile([0.25, 0.75])

    x_iqr = x_q3 - x_q1
    y_iqr = y_q3 - y_q1

    if x_iqr == 0 or y_iqr == 0:
        return pd.Series(False, index=df.index)

    mask = (
        (df[xcol] < x_q1 - 1.5 * x_iqr) |
        (df[xcol] > x_q3 + 1.5 * x_iqr) |
        (df[ycol] < y_q1 - 1.5 * y_iqr) |
        (df[ycol] > y_q3 + 1.5 * y_iqr)
    )
    return mask.fillna(False)


def plot_jointplots_por_pares(df_month, years=(2021, 2022, 2023, 2024, 2025)):
    pares = preparar_pares_mensuales(df_month, years=years)
    pares_unicos = sorted(set((g1, g2) for g1, g2, _, _ in pares))
    metricas = ["viajeros", "pernoctaciones", "alos"]

    if not pares_unicos:
        return None

    n_rows = len(pares_unicos)

    if n_rows <= 1:
        vertical_spacing = 0.08
    else:
        max_allowed = 1 / (n_rows - 1)
        vertical_spacing = min(0.02, max_allowed * 0.85)

    fig = make_subplots(
        rows=n_rows,
        cols=3,
        subplot_titles=[
            f"{g1} vs {g2} - {NOMBRES_METRICAS[m]}"
            for g1, g2 in pares_unicos
            for m in metricas
        ],
        horizontal_spacing=0.06,
        vertical_spacing=vertical_spacing
    )

    mapa_color_anio = {
        2021: "#1f77b4",
        2022: "#ff7f0e",
        2023: "#2ca02c",
        2024: "#d62728",
        2025: "#9467bd"
    }

    for row_idx, (g1, g2) in enumerate(pares_unicos, start=1):
        for col_idx, metric in enumerate(metricas, start=1):
            pair_df = None
            for a, b, m, subdf in pares:
                if a == g1 and b == g2 and m == metric:
                    pair_df = subdf.copy()
                    break

            if pair_df is None:
                continue

            xcol = f"{metric}__{g1}"
            ycol = f"{metric}__{g2}"
            pair_df["anomalia"] = detectar_anomalias_iqr(pair_df, xcol, ycol)

            for anio in sorted(pair_df["año"].dropna().unique()):
                tmp = pair_df[pair_df["año"] == anio].copy()

                if tmp.empty:
                    continue

                fig.add_trace(
                    go.Scatter(
                        x=tmp[xcol],
                        y=tmp[ycol],
                        mode="markers",
                        name=str(anio),
                        legendgroup=str(anio),
                        showlegend=(row_idx == 1 and col_idx == 1),
                        marker=dict(
                            size=np.where(tmp["anomalia"], 12, 8),
                            color=mapa_color_anio.get(int(anio), COLOR_GRIS),
                            line=dict(
                                color=np.where(tmp["anomalia"], "black", mapa_color_anio.get(int(anio), COLOR_GRIS)),
                                width=np.where(tmp["anomalia"], 1.4, 0)
                            ),
                            opacity=0.80
                        ),
                        text=tmp["fecha"].dt.strftime("%Y-%m"),
                        customdata=np.column_stack([
                            np.where(tmp["anomalia"], "Sí", "No")
                        ]),
                        hovertemplate=(
                            f"{g1}: %{{x:,.2f}}<br>"
                            f"{g2}: %{{y:,.2f}}<br>"
                            "Periodo: %{text}<br>"
                            f"Métrica: {NOMBRES_METRICAS[metric]}<br>"
                            "Posible anomalía: %{customdata[0]}<extra></extra>"
                        )
                    ),
                    row=row_idx,
                    col=col_idx
                )

            fig.update_xaxes(title_text=g1, row=row_idx, col=col_idx)
            fig.update_yaxes(title_text=g2, row=row_idx, col=col_idx)

    fig.update_layout(
        template="plotly_white",
        height=max(320 * n_rows, 900),
        title="Relación mensual entre ubicaciones por viajeros, pernoctaciones y estancia media (2021-2025)",
        legend_title_text="Año",
        margin=dict(l=40, r=30, t=90, b=80)
    )

    agregar_nota_abreviaturas(
        fig,
        texto_extra="Method note: black-outlined larger points flag possible anomalies using an IQR rule.",
        yshift=-0.04
    )
    return fig


# =========================================================
# 4) HEATMAPS DE ESTACIONALIDAD
# =========================================================
def plot_heatmap_estacionalidad(df, metric, title, log_scale=False, escala="Blues"):
    month_order = list(range(1, 13))
    nombres_meses = [traducir_mes_num_a_es()[m] for m in month_order]

    heat = (
        df.pivot(index="geo_name", columns="mes", values=metric)
        .reindex(columns=month_order)
    )

    z = np.log1p(heat) if log_scale else heat
    color_label = f"log(1+{NOMBRES_METRICAS.get(metric, metric)})" if log_scale else NOMBRES_METRICAS.get(metric, metric)

    fig = px.imshow(
        z,
        x=nombres_meses,
        y=heat.index.tolist(),
        aspect="auto",
        color_continuous_scale=escala,
        labels=dict(
            x="Mes",
            y="Geografía",
            color=color_label
        ),
        title=title
    )

    aplicar_layout_estandar(fig, height=520)
    fig.update_xaxes(side="bottom")
    return fig


# =========================================================
# 5) TENDENCIAS POR GEO CON BANDA MIN-MAX ANUAL + ALOS EN EXTREMOS
# =========================================================
def build_yearly_min_max(df_geo):
    resumen = (
        df_geo.groupby("año", as_index=False)
        .agg(
            min_pernoctaciones=("pernoctaciones", "min"),
            max_pernoctaciones=("pernoctaciones", "max")
        )
    )
    return resumen


def extract_extreme_points_with_alos(df_geo):
    rows = []
    for year, tmp in df_geo.groupby("año"):
        tmp_valid = tmp.dropna(subset=["pernoctaciones", "alos", "fecha"]).copy()
        if tmp_valid.empty:
            continue

        idx_min = tmp_valid["pernoctaciones"].idxmin()
        idx_max = tmp_valid["pernoctaciones"].idxmax()

        min_row = tmp_valid.loc[idx_min].copy()
        max_row = tmp_valid.loc[idx_max].copy()

        rows.append({
            "año": year,
            "tipo": "mínimo",
            "fecha": min_row["fecha"],
            "pernoctaciones": min_row["pernoctaciones"],
            "alos": min_row["alos"]
        })
        rows.append({
            "año": year,
            "tipo": "máximo",
            "fecha": max_row["fecha"],
            "pernoctaciones": max_row["pernoctaciones"],
            "alos": max_row["alos"]
        })

    return pd.DataFrame(rows)


def plot_tendencia_geo_con_banda(df, geo_name, metric="pernoctaciones", title=None):
    tmp = (
        df[df["geo_name"] == geo_name]
        .copy()
        .sort_values("fecha")
    )

    tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")
    tmp[metric] = pd.to_numeric(tmp[metric], errors="coerce")
    tmp["alos"] = pd.to_numeric(tmp["alos"], errors="coerce")
    tmp = tmp.dropna(subset=["fecha", metric, "año"]).copy()

    if tmp.empty:
        return go.Figure()

    yearly_band = build_yearly_min_max(tmp)
    yearly_band["fecha_anchor"] = pd.to_datetime(yearly_band["año"].astype(str) + "-07-01", errors="coerce")

    extremos = extract_extreme_points_with_alos(tmp)

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=yearly_band["fecha_anchor"],
            y=yearly_band["max_pernoctaciones"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
            name="Máximo anual"
        )
    )
    fig.add_trace(
        go.Scatter(
            x=yearly_band["fecha_anchor"],
            y=yearly_band["min_pernoctaciones"],
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(31,119,180,0.18)",
            line=dict(width=0),
            name="Rango anual mín-máx",
            hovertemplate=(
                "Año ancla: %{x|%Y}<br>"
                "Mínimo anual: %{y:,.0f}<extra></extra>"
            )
        )
    )

    fig.add_trace(
        go.Scatter(
            x=tmp["fecha"],
            y=tmp[metric],
            mode="lines+markers",
            name=NOMBRES_METRICAS.get(metric, metric),
            line=dict(color=COLOR_PRIMARIO, width=2),
            marker=dict(size=6)
        )
    )

    for _, r in extremos.iterrows():
        fig.add_trace(
            go.Scatter(
                x=[r["fecha"]],
                y=[r["pernoctaciones"]],
                mode="markers+text",
                text=[f"ALOS: {r['alos']:.2f}"],
                textposition="top center" if r["tipo"] == "máximo" else "bottom center",
                marker=dict(
                    size=10,
                    color=COLOR_CUARTO if r["tipo"] == "máximo" else COLOR_TERCIARIO
                ),
                name=f"{r['tipo'].capitalize()} anual",
                showlegend=False,
                hovertemplate=(
                    f"{r['tipo'].capitalize()} anual<br>"
                    "Fecha: %{x|%Y-%m}<br>"
                    "Pernoctaciones: %{y:,.0f}<br>"
                    f"ALOS: {r['alos']:.2f}<extra></extra>"
                )
            )
        )

    fig.update_xaxes(title="Fecha", tickformat="%Y-%m")
    fig.update_yaxes(title="Pernoctaciones", separatethousands=True)
    aplicar_layout_estandar(
        fig,
        height=540,
        titulo=title or f"Tendencia mensual de pernoctaciones con banda anual mínimo-máximo - {geo_name}"
    )
    agregar_nota_abreviaturas(fig, yshift=-0.22)
    return fig


# =========================================================
# 6) WAFFLE CHART POR TIPO DE ALOJAMIENTO + TABLA
# =========================================================
def make_tabla_resumen_por_alojamiento(df_aloj, tipo_alojamiento):
    tmp = df_aloj[df_aloj["tipo_alojamiento"] == tipo_alojamiento].copy()

    tabla = (
        tmp.groupby("geo_name", as_index=False)
        .agg(
            min_ons=("pernoctaciones", "min"),
            max_ons=("pernoctaciones", "max"),
            avg_ons=("pernoctaciones", "mean"),
            avg_tourists=("viajeros", "mean"),
            avg_alos=("alos", "mean")
        )
        .rename(columns={
            "geo_name": "Location",
            "min_ons": "Min OnS",
            "max_ons": "Max OnS",
            "avg_ons": "Avg OnS",
            "avg_tourists": "Avg Tourists",
            "avg_alos": "Avg ALOS"
        })
        .sort_values("Avg OnS", ascending=False)
    )

    for col in ["Min OnS", "Max OnS", "Avg OnS", "Avg Tourists", "Avg ALOS"]:
        tabla[col] = pd.to_numeric(tabla[col], errors="coerce")

    return tabla


def build_waffle_grid(values_by_geo, rows=10, cols=10):
    total_cells = rows * cols
    total_value = values_by_geo.sum()

    if total_value <= 0 or values_by_geo.empty:
        return pd.DataFrame(columns=["x", "y", "geo_name", "valor"])

    proportions = values_by_geo / total_value
    raw_cells = proportions * total_cells
    base_cells = np.floor(raw_cells).astype(int)
    remainder = raw_cells - base_cells

    cells_assigned = base_cells.sum()
    missing = total_cells - cells_assigned

    if missing > 0:
        for idx in remainder.sort_values(ascending=False).index[:missing]:
            base_cells.loc[idx] += 1
    elif missing < 0:
        for idx in remainder.sort_values(ascending=True).index[:abs(missing)]:
            if base_cells.loc[idx] > 0:
                base_cells.loc[idx] -= 1

    records = []
    counter = 0
    for geo, n in base_cells.items():
        for _ in range(n):
            r = counter // cols
            c = counter % cols
            records.append({"x": c + 1, "y": rows - r, "geo_name": geo, "valor": values_by_geo.loc[geo]})
            counter += 1

    return pd.DataFrame(records)


def plot_waffle_por_tipo_alojamiento(df_aloj, tipo_alojamiento, metric="pernoctaciones", rows=10, cols=10):
    tmp = df_aloj[df_aloj["tipo_alojamiento"] == tipo_alojamiento].copy()

    agg = (
        tmp.groupby("geo_name", as_index=True)[metric]
        .sum()
        .sort_values(ascending=False)
    )

    grid = build_waffle_grid(agg, rows=rows, cols=cols)
    tabla = make_tabla_resumen_por_alojamiento(df_aloj, tipo_alojamiento)

    paleta = px.colors.qualitative.Set2 + px.colors.qualitative.Safe + px.colors.qualitative.Pastel
    color_map = {geo: paleta[i % len(paleta)] for i, geo in enumerate(agg.index)}

    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.58, 0.42],
        specs=[[{"type": "scatter"}, {"type": "table"}]],
        horizontal_spacing=0.05,
        subplot_titles=(
            f"Contribución por ubicación - {tipo_alojamiento}",
            "Resumen estadístico por ubicación"
        )
    )

    for geo in agg.index:
        sub = grid[grid["geo_name"] == geo].copy()
        fig.add_trace(
            go.Scatter(
                x=sub["x"],
                y=sub["y"],
                mode="markers",
                name=geo,
                marker=dict(
                    symbol="square",
                    size=26,
                    color=color_map[geo],
                    line=dict(color="white", width=1)
                ),
                customdata=np.column_stack([
                    np.repeat(geo, len(sub)),
                    np.repeat(agg.loc[geo], len(sub))
                ]),
                hovertemplate=(
                    "Ubicación: %{customdata[0]}<br>"
                    f"{metric.capitalize()}: %{{customdata[1]:,.0f}}<extra></extra>"
                ),
                showlegend=True
            ),
            row=1, col=1
        )

    fig.add_trace(
        go.Table(
            header=dict(
                values=list(tabla.columns),
                fill_color="#E8EEF7",
                align="left"
            ),
            cells=dict(
                values=[
                    tabla["Location"],
                    tabla["Min OnS"].map(lambda x: f"{x:,.0f}" if pd.notna(x) else ""),
                    tabla["Max OnS"].map(lambda x: f"{x:,.0f}" if pd.notna(x) else ""),
                    tabla["Avg OnS"].map(lambda x: f"{x:,.0f}" if pd.notna(x) else ""),
                    tabla["Avg Tourists"].map(lambda x: f"{x:,.0f}" if pd.notna(x) else ""),
                    tabla["Avg ALOS"].map(lambda x: f"{x:,.2f}" if pd.notna(x) else "")
                ],
                align="left"
            )
        ),
        row=1, col=2
    )

    fig.update_xaxes(
        title="Columnas de la rejilla",
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        row=1, col=1
    )
    fig.update_yaxes(
        title="Filas de la rejilla",
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor="x",
        scaleratio=1,
        row=1, col=1
    )

    fig.update_layout(
        template="plotly_white",
        height=720,
        title=f"Gráfico waffle por tipo de alojamiento - {tipo_alojamiento} (2021-2025)",
        margin=dict(l=40, r=30, t=90, b=80),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        )
    )

    agregar_nota_abreviaturas(
        fig,
        texto_extra="Nota metodológica: cada cuadrado representa una parte proporcional del total acumulado del periodo 2021-2025.",
        yshift=-0.18
    )

    return fig, tabla


# =========================================================
# BLOQUE PRINCIPAL DE EJECUCIÓN
# =========================================================
# REQUISITOS ESPERADOS EN EL ENTORNO:
# - kpi_geo: DataFrame con columnas al menos ["fecha", "geo_name", "viajeros", "pernoctaciones"]
# - kpi_aloj: DataFrame con columnas al menos ["fecha", "geo_name", "tipo_alojamiento", "viajeros", "pernoctaciones"]
# - TARGET_GEOS: lista de geografías objetivo
# - FOCUS_COMMUNITIES: lista de ubicaciones foco para alojamiento
#
# Si ya tienes definidas safe_divide, add_time_features, add_growth_features o latest_12m
# en tu notebook/script, este bloque usa las versiones locales definidas arriba.


# -------------------------
# 0. Construcción de bases
# -------------------------
kpi_geo_plot = build_geo_month_base(kpi_geo, TARGET_GEOS)
efficiency_year = build_geo_year_base(kpi_geo_plot, years=(2021, 2022, 2023, 2024, 2025))
geo_month_avg = build_seasonality_base(kpi_geo_plot)
aloj_base = build_accommodation_base(kpi_aloj, FOCUS_COMMUNITIES, years=(2021, 2022, 2023, 2024, 2025))

print("Geografías en base mensual:", sorted(kpi_geo_plot["geo_name"].unique()))
print(kpi_geo_plot.groupby("geo_name").size())

figures_to_show = []
tablas_generadas = {}


# -------------------------
# 1. Barras 5 años por geografía
# -------------------------
orden_geos_viajeros = ordenar_geos_por_12m(kpi_geo_plot, "viajeros")
orden_geos_pernoct = ordenar_geos_por_12m(kpi_geo_plot, "pernoctaciones")

fig_market_viajeros_5y = plot_barras_5_anios_por_geo(
    efficiency_year,
    orden_geos=orden_geos_viajeros,
    metric="viajeros",
    title="Viajeros de los últimos 5 años por geografía objetivo"
)
figures_to_show.append(fig_market_viajeros_5y)

fig_market_pernoct_5y = plot_barras_5_anios_por_geo(
    efficiency_year,
    orden_geos=orden_geos_pernoct,
    metric="pernoctaciones",
    title="Pernoctaciones de los últimos 5 años por geografía objetivo"
)
figures_to_show.append(fig_market_pernoct_5y)


# -------------------------
# 2. Trayectoria anual 2023-2025 + tabla
# -------------------------
fig_trayectoria_yoy_alos, tabla_yoy = plot_trayectoria_alos_yoy_con_tabla(
    efficiency_year,
    years_plot=(2023, 2024, 2025),
    yoy_cap=200
)
figures_to_show.append(fig_trayectoria_yoy_alos)
tablas_generadas["tabla_resumen_yoy_2022_vs_2023_2025"] = tabla_yoy
print_tabla_titulo("Tabla resumen YoY 2022 vs promedio 2023-2025", tabla_yoy)


# -------------------------
# 3. Jointplots mensuales 2021-2025 por pares
# -------------------------
fig_jointplots = plot_jointplots_por_pares(
    kpi_geo_plot,
    years=(2021, 2022, 2023, 2024, 2025)
)
if fig_jointplots is not None:
    figures_to_show.append(fig_jointplots)


# -------------------------
# 4. Heatmap de estacionalidad de pernoctaciones
# -------------------------
fig_heatmap_pernoct = plot_heatmap_estacionalidad(
    geo_month_avg,
    metric="avg_pernoctaciones",
    title="Mapa de calor de estacionalidad de pernoctaciones medias mensuales",
    log_scale=True,
    escala="Blues"
)
figures_to_show.append(fig_heatmap_pernoct)


# -------------------------
# 5. Heatmap de estacionalidad de ALOS
# -------------------------
fig_heatmap_alos = plot_heatmap_estacionalidad(
    geo_month_avg,
    metric="avg_alos",
    title="Mapa de calor de estacionalidad de estancia media mensual",
    log_scale=False,
    escala="Viridis"
)
figures_to_show.append(fig_heatmap_alos)


# -------------------------
# 6. Tendencias por geografía con banda anual y ALOS en extremos
# -------------------------
for g in sorted(kpi_geo_plot["geo_name"].unique()):
    fig_trend = plot_tendencia_geo_con_banda(
        df=kpi_geo_plot,
        geo_name=g,
        metric="pernoctaciones",
        title=f"Tendencia mensual de pernoctaciones con banda anual mínimo-máximo - {g}"
    )
    figures_to_show.append(fig_trend)


# -------------------------
# 7. Waffle charts por tipo de alojamiento + tablas
# -------------------------
tipos_alojamiento = sorted(aloj_base["tipo_alojamiento"].dropna().unique())

for tipo in tipos_alojamiento:
    fig_waffle, tabla_waffle = plot_waffle_por_tipo_alojamiento(
        aloj_base,
        tipo_alojamiento=tipo,
        metric="pernoctaciones",
        rows=10,
        cols=10
    )
    figures_to_show.append(fig_waffle)

    key = f"tabla_waffle_{tipo}"
    tablas_generadas[key] = tabla_waffle
    print_tabla_titulo(f"Tabla resumen para tipo de alojamiento: {tipo}", tabla_waffle)


# -------------------------
# 8. Mostrar figuras
# -------------------------
for fig in figures_to_show:
    fig.show()


# -------------------------
# 9. Exportación opcional HTML
# -------------------------
EXPORTAR_HTML = False

if EXPORTAR_HTML:
    for i, fig in enumerate(figures_to_show, start=1):
        guardar_html(fig, f"figura_{i:02d}.html")
    print(f"HTML exportados en: {OUTPUT_DIR.resolve()}")

Geografías en base mensual: ['Balears, Illes', 'Barcelona', 'Girona', 'Madrid', 'Málaga', 'Sevilla', 'Valencia/València']
geo_name
Balears, Illes       60
Barcelona            60
Girona               60
Madrid               60
Málaga               60
Sevilla              60
Valencia/València    60
dtype: int64

Tabla resumen YoY 2022 vs promedio 2023-2025
  periodo         ubicacion  valor
     2022    Balears, Illes 114.96
2023-2025    Balears, Illes   4.62
     2022         Barcelona 134.39
2023-2025         Barcelona   4.74
     2022            Girona  72.31
2023-2025            Girona   4.94
     2022            Madrid  75.91
2023-2025            Madrid   7.58
     2022            Málaga  80.77
2023-2025            Málaga   4.62
     2022           Sevilla  91.75
2023-2025           Sevilla   5.83
     2022 Valencia/València  63.95
2023-2025 Valencia/València   3.39

Tabla resumen para tipo de alojamiento: Encuesta de Ocupación Hotelera
            Location  Min OnS    Max OnS     

In [39]:
print("kpi_aloj shape:", kpi_aloj.shape)
print("\nColumns:", kpi_aloj.columns.tolist())

print("\nUnique geo_name sample:")
print(sorted(kpi_aloj["geo_name"].dropna().astype(str).unique())[:50])

print("\nRows by geo_name:")
print(kpi_aloj.groupby("geo_name").size().sort_values(ascending=False).head(30))

print("\nTarget geos present in kpi_aloj:")
print(sorted(set(kpi_aloj["geo_name"].dropna().unique()).intersection(set(TARGET_GEOS))))

print("\nLatest 12M rows by geo_name:")
tmp = latest_12m(kpi_aloj).copy()
print(tmp.groupby("geo_name").size().sort_values(ascending=False).head(30))

kpi_aloj shape: (17100, 40)

Columns: ['fecha', 'geo_level', 'geo_name', 'nombre_comunidad', 'tipo_alojamiento', 'residencia_group', 'residencia_detail', 'pernoctaciones', 'viajeros', 'alos', 'año', 'mes', 'quarter', 'month_name', 'month_id', 'lag1_viajeros', 'lag12_viajeros', 'mom_viajeros', 'yoy_viajeros', 'roll3_viajeros', 'roll12_viajeros', 'lag1_pernoctaciones', 'lag12_pernoctaciones', 'mom_pernoctaciones', 'yoy_pernoctaciones', 'roll3_pernoctaciones', 'roll12_pernoctaciones', 'lag1_alos', 'lag12_alos', 'mom_alos', 'yoy_alos', 'roll3_alos', 'roll12_alos', 'viajeros_total_mes', 'pernoctaciones_total_mes', 'share_viajeros', 'share_pernoctaciones', 'seasonality_idx_viajeros', 'seasonality_idx_pernoctaciones', 'seasonality_idx_alos']

Unique geo_name sample:
['Andalucía', 'Aragón', 'Asturias, Principado de', 'Balears, Illes', 'Canarias', 'Cantabria', 'Castilla - La Mancha', 'Castilla y León', 'Cataluña', 'Ceuta', 'Comunitat Valenciana', 'Extremadura', 'Galicia', 'Madrid, Comunidad de'

___
___
___
